# Определение занятости парковочного места с использованием нейронных сетей

**Тема:** Бинарная классификация изображений парковочных мест (Empty / Occupied)

**Датасет:** PKLot (Voxel51/PKLot)

**Архитектуры:**
1. ResNet18
2. DenseNet121
3. EfficientNet-B0
4. MobileNetV3-Large
5. Vision Transformer (ViT-B/16)

In [ ]:
# Установка библиотек (torch и torchvision уже предустановлены в Colab)
!pip install -q Pillow==9.5.0
!pip install -q kagglehub openpyxl tqdm scikit-learn matplotlib pandas numpy

In [ ]:
# Проверка доступности GPU
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найдена. Перейдите: Среда выполнения → Сменить среду выполнения → GPU")

In [ ]:
# Создание директорий проекта
import os

for d in ['saved_models', 'plots', 'results', 'data', 'models']:
    os.makedirs(d, exist_ok=True)

print("Директории созданы")

## Исходный код модулей

Ниже записываются файлы проекта с помощью команды `%%writefile`.

In [ ]:
%%writefile config.py
"""
config.py — Центральная конфигурация классификатора занятости парковочных мест.

Все константы проекта, пути, гиперпараметры, имена классов и трансформации данных
определены здесь, чтобы каждый другой модуль мог импортировать из единого источника.

Использование
-------------
    from config import Config, get_train_transforms, get_val_transforms, seed_everything

    device = Config.DEVICE          # torch.device("cuda") / "mps" / "cpu"
    names  = Config.CLASS_NAMES     # ["Empty", "Occupied"]
    train_tf = get_train_transforms()
"""

import logging
import os
import random
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torchvision import transforms

# ---------------------------------------------------------------------------
# Логгер модуля
# ---------------------------------------------------------------------------
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Определение устройства при импорте
# ---------------------------------------------------------------------------
def _resolve_device() -> torch.device:
    """Возвращает наилучшее доступное вычислительное устройство: CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


# ---------------------------------------------------------------------------
# Config — все константы как атрибуты класса
# ---------------------------------------------------------------------------
class Config:
    """
    Центральная конфигурация проекта.

    Все атрибуты определены на уровне класса, чтобы каждый модуль мог
    обращаться к ним без создания экземпляра: ``Config.DEVICE``,
    ``Config.CLASS_NAMES`` и т.д.
    """

    # ------------------------------------------------------------------
    # Воспроизводимость
    # ------------------------------------------------------------------
    SEED: int = 42

    # ------------------------------------------------------------------
    # Данные
    # ------------------------------------------------------------------
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_CLASSES: int = 2
    CLASS_NAMES: tuple[str, ...] = ("Empty", "Occupied")

    # ------------------------------------------------------------------
    # Гиперпараметры обучения
    # ------------------------------------------------------------------
    NUM_EPOCHS: int = 20
    LEARNING_RATE: float = 0.0001

    #: EarlyStopping: остановка после заданного числа эпох без улучшения val-loss
    PATIENCE: int = 5

    # ------------------------------------------------------------------
    # Вычислительное устройство (определяется один раз при импорте)
    # ------------------------------------------------------------------
    DEVICE: torch.device = _resolve_device()

    # ------------------------------------------------------------------
    # Статистики нормализации ImageNet
    # ------------------------------------------------------------------
    IMAGENET_MEAN: tuple[float, ...] = (0.485, 0.456, 0.406)
    IMAGENET_STD: tuple[float, ...] = (0.229, 0.224, 0.225)

    # ------------------------------------------------------------------
    # Имена архитектур (должны совпадать с ключами в models.py)
    # ------------------------------------------------------------------
    MODEL_NAMES: tuple[str, ...] = (
        "ResNet18",
        "DenseNet121",
        "EfficientNet-B0",
        "MobileNetV3-Large",
        "ViT-B/16",
    )

    # ------------------------------------------------------------------
    # Пути файловой системы (все относительно директории этого файла)
    # ------------------------------------------------------------------
    #: Корневая директория проекта (директория, содержащая config.py)
    PROJECT_ROOT: Path = Path(__file__).resolve().parent

    #: Корневая директория датасета — сюда попадут изображения PKLot
    DATA_DIR: Path = Path(__file__).resolve().parent / "data"

    #: Структура разбиений в стиле torchvision:
    #:   models/train/{Empty,Occupied}/…
    #:   models/val/{Empty,Occupied}/…
    #:   models/test/{Empty,Occupied}/…
    MODELS_DIR: Path = Path(__file__).resolve().parent / "models"

    #: Контрольные точки лучших эпох (.pth файлы)
    SAVED_MODELS_DIR: Path = Path(__file__).resolve().parent / "saved_models"

    #: Метрики обучения и оценки: comparison.csv, comparison.xlsx, JSON
    RESULTS_DIR: Path = Path(__file__).resolve().parent / "results"

    #: Все PNG-графики (кривые потерь, матрицы ошибок, ROC-кривые)
    PLOTS_DIR: Path = Path(__file__).resolve().parent / "plots"

    #: История предсказаний Streamlit
    HISTORY_FILE: Path = Path(__file__).resolve().parent / "history.json"


# ---------------------------------------------------------------------------
# Создание необходимых директорий при импорте
# ---------------------------------------------------------------------------
for _directory in (
    Config.DATA_DIR,
    Config.MODELS_DIR,
    Config.SAVED_MODELS_DIR,
    Config.RESULTS_DIR,
    Config.PLOTS_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)

logger.info(
    "Config loaded — device: %s | classes: %s",
    Config.DEVICE,
    Config.CLASS_NAMES,
)


# ---------------------------------------------------------------------------
# Утилита: воспроизводимость
# ---------------------------------------------------------------------------
def seed_everything(seed: int) -> None:
    """
    Фиксирует все случайные зерна для полной воспроизводимости.

    Устанавливает зерна для встроенного модуля Python ``random``, NumPy,
    PyTorch (как для CPU, так и для каждого CUDA-устройства) и включает
    детерминированный режим cuDNN.

    Параметры
    ----------
    seed:
        Целочисленное значение зерна. Передавайте ``Config.SEED`` (42)
        во всём проекте для обеспечения согласованных результатов между
        запусками.

    Примечания
    ----------
    ``torch.backends.cudnn.benchmark`` отключается, чтобы на каждом
    прямом проходе выбирался один и тот же алгоритм свёртки, даже
    ценой небольшой потери производительности.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info("Random seed fixed to %d", seed)


# ---------------------------------------------------------------------------
# Трансформации данных
# ---------------------------------------------------------------------------
def get_train_transforms() -> transforms.Compose:
    """
    Возвращает пайплайн аугментации, применяемый к обучающим изображениям.

    Все операции аугментации применяются последовательно до передачи
    тензора в модель.

    Пайплайн
    --------
    1. ``RandomResizedCrop(224, scale=(0.8, 1.0))``
       Случайная обрезка, сохраняющая 80–100% исходной площади, с
       последующим масштабированием до 224×224.
    2. ``RandomHorizontalFlip()``
       Горизонтальное отражение с вероятностью 0.5.
    3. ``RandomRotation(15)``
       Случайный поворот в диапазоне [−15°, +15°].
    4. ``ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)``
       Случайное изменение цветовых характеристик для фотометрической инвариантности.
    5. ``ToTensor()``
       Преобразует PIL-изображение (H×W×C, uint8) в тензор float32 (C×H×W)
       в диапазоне [0, 1].
    6. ``Normalize(IMAGENET_MEAN, IMAGENET_STD)``
       Пополосная стандартизация по статистикам ImageNet.

    Возвращает
    ----------
    transforms.Compose
        Составной трансформ, готовый для передачи в ``torch.utils.data.Dataset``.
    """
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(
                Config.IMAGE_SIZE,
                scale=(0.8, 1.0),
            ),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1,
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=Config.IMAGENET_MEAN,
                std=Config.IMAGENET_STD,
            ),
        ]
    )


def get_val_transforms() -> transforms.Compose:
    """
    Возвращает детерминированный пайплайн предобработки для изображений
    валидации и тестирования.

    Аугментация здесь не применяется — только масштабирование и
    нормализация, необходимые для воспроизведения входной статистики
    времени обучения.

    Пайплайн
    --------
    1. ``Resize(256)``
       Масштабирование короткой стороны до 256 пикселей с сохранением
       соотношения сторон.
    2. ``CenterCrop(224)``
       Извлечение центрального патча 224×224.
    3. ``ToTensor()``
       Преобразует PIL-изображение (H×W×C, uint8) в тензор float32 (C×H×W)
       в диапазоне [0, 1].
    4. ``Normalize(IMAGENET_MEAN, IMAGENET_STD)``
       Пополосная стандартизация по статистикам ImageNet.

    Возвращает
    ----------
    transforms.Compose
        Составной трансформ, готовый для передачи в ``torch.utils.data.Dataset``.
    """
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(Config.IMAGE_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=Config.IMAGENET_MEAN,
                std=Config.IMAGENET_STD,
            ),
        ]
    )


In [ ]:
%%writefile dataset.py
"""
dataset.py — Загрузка, подготовка датасета PKLot и создание DataLoader.

Этот модуль обрабатывает все этапы подготовки данных для классификатора
занятости парковочных мест:

1. Загрузка датасета PKLot с Kaggle через kagglehub.
2. Разбиение по парковкам:
   - train / val — PUCPR + UFPR04 (все погоды), стратифицированное 85 / 15.
   - test — UFPR05 (все погоды).
3. Копирование в структуру директорий ``torchvision.datasets.ImageFolder``::

       data_dir/
           train/
               Empty/
               Occupied/
           val/
               Empty/
               Occupied/
           test/
               Empty/
               Occupied/

4. Создание экземпляров PyTorch ``DataLoader``, готовых для обучения.

Публичное API
-------------
- ``DatasetInfo``                — датакласс с размерами разбиений и числом классов
- ``download_and_prepare_dataset(data_dir)`` — идемпотентная функция подготовки
- ``get_data_loaders(...)``      — возвращает (train_loader, val_loader, test_loader)

Использование
-------------
    from dataset import download_and_prepare_dataset, get_data_loaders, DatasetInfo
    from config import Config, get_train_transforms, get_val_transforms

    data_root = download_and_prepare_dataset(Config.DATA_DIR)
    train_dl, val_dl, test_dl = get_data_loaders(
        data_root, Config.BATCH_SIZE,
        get_train_transforms(), get_val_transforms(),
    )
"""

from __future__ import annotations

import logging
import shutil
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import datasets

# ---------------------------------------------------------------------------
# Логгер модуля
# ---------------------------------------------------------------------------
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Константы
# ---------------------------------------------------------------------------

# Идентификатор датасета на Kaggle.
_KAGGLE_DATASET: str = "blanderbuss/parking-lot-dataset"

# Парковки для обучения и тестирования.
_TRAIN_LOTS: tuple[str, ...] = ("PUCPR", "UFPR04")
_TEST_LOT: str = "UFPR05"

# Допустимые расширения изображений (строчные).
_IMAGE_EXTENSIONS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})

_SPLIT_NAMES: tuple[str, ...] = ("train", "val", "test")
_CLASS_NAMES: tuple[str, ...] = ("Empty", "Occupied")

# Доля валидации внутри обучающих парковок (PUCPR + UFPR04).
_VAL_RATIO: float = 0.15

# Настройки DataLoader
_NUM_WORKERS: int = 2


# ---------------------------------------------------------------------------
# Датакласс DatasetInfo
# ---------------------------------------------------------------------------
@dataclass
class DatasetInfo:
    """
    Сводная статистика по подготовленному датасету PKLot.

    Атрибуты
    ----------
    num_train:
        Количество изображений в обучающем разбиении.
    num_val:
        Количество изображений в валидационном разбиении.
    num_test:
        Количество изображений в тестовом разбиении.
    class_distribution:
        Вложенное отображение ``{имя_разбиения: {имя_класса: количество}}``
        с абсолютным числом изображений по классам для каждого разбиения.
    """

    num_train: int
    num_val: int
    num_test: int
    class_distribution: dict[str, dict[str, int]] = field(default_factory=dict)

    @property
    def total(self) -> int:
        """Общее количество изображений по всем разбиениям."""
        return self.num_train + self.num_val + self.num_test

    def log_summary(self) -> None:
        """Записывает читаемую сводку в логгер модуля."""
        logger.info("Сводка датасета — всего изображений: %d", self.total)
        logger.info(
            "  train=%d | val=%d | test=%d",
            self.num_train,
            self.num_val,
            self.num_test,
        )
        for split, counts in self.class_distribution.items():
            parts = ", ".join(f"{cls}={n}" for cls, n in sorted(counts.items()))
            logger.info("  %s: %s", split, parts)


# ---------------------------------------------------------------------------
# Внутренние вспомогательные функции
# ---------------------------------------------------------------------------

def _sentinel_path(data_dir: Path) -> Path:
    """Возвращает путь к файлу-сторожу, отмечающему завершённую подготовку."""
    return data_dir / ".pklot_prepared"


def _is_already_prepared(data_dir: Path) -> bool:
    """
    Возвращает ``True``, если структура ImageFolder уже была построена
    в предыдущем запуске.

    Функция проверяет как наличие файла-сторожа, записанного в конце
    подготовки, так и то, что все ожидаемые директории разбиений/классов
    существуют.
    """
    if not _sentinel_path(data_dir).exists():
        return False
    for split in _SPLIT_NAMES:
        for cls in _CLASS_NAMES:
            if not (data_dir / split / cls).is_dir():
                return False
    return True


def _build_empty_imagefolder_tree(data_dir: Path) -> None:
    """
    Создаёт пустое дерево директорий для структуры ImageFolder.

    Создаёт ``data_dir/{разбиение}/{класс}/`` для каждой комбинации
    имени разбиения и имени класса. Существующие директории не затрагиваются.
    """
    for split in _SPLIT_NAMES:
        for cls in _CLASS_NAMES:
            target = data_dir / split / cls
            target.mkdir(parents=True, exist_ok=True)
            logger.debug("Директория готова: %s", target)


def _download_kaggle(dataset_id: str) -> Path:
    """
    Загружает датасет PKLot с Kaggle и возвращает путь к корню
    с парковками (директория, содержащая PUCPR / UFPR04 / UFPR05).
    """
    import kagglehub  # type: ignore[import]

    logger.info("Загрузка датасета '%s' с Kaggle …", dataset_id)
    raw_path = Path(kagglehub.dataset_download(dataset_id))
    logger.info("Датасет загружен: %s", raw_path)

    root = _find_image_root(raw_path)
    logger.info("Корень изображений: %s", root)
    return root


def _find_image_root(base: Path) -> Path:
    """
    Рекурсивно ищет директорию, содержащую папки парковок
    (PUCPR, UFPR04, UFPR05). Kaggle может вложить данные на
    произвольную глубину.
    """
    lot_names = set(_TRAIN_LOTS) | {_TEST_LOT}

    if lot_names.issubset({d.name for d in base.iterdir() if d.is_dir()}):
        return base

    for child in sorted(base.iterdir()):
        if not child.is_dir():
            continue
        try:
            subdirs = {d.name for d in child.iterdir() if d.is_dir()}
        except PermissionError:
            continue
        if lot_names.issubset(subdirs):
            return child
        found = _find_image_root(child)
        if found != child:
            return found

    raise RuntimeError(
        f"Не удалось найти директорию с парковками {lot_names} "
        f"внутри {base}. Проверьте структуру загруженного датасета."
    )


def _collect_all_images(
    extracted_root: Path,
) -> list[tuple[Path, str, str]]:
    """
    Обходит дерево директорий PKLotSegmented и собирает все изображения.

    Ожидаемая структура::

        PKLotSegmented/
            {parking_lot}/{weather}/{date}/Empty/*.jpg
            {parking_lot}/{weather}/{date}/Occupied/*.jpg

    Параметры
    ----------
    extracted_root:
        Корень распакованного архива PKLotSegmented.

    Возвращает
    ----------
    list of (src_path, class_name, unique_filename)
        ``class_name`` — ``"Empty"`` или ``"Occupied"``.
        ``unique_filename`` — уникальное имя файла вида
        ``{parking_lot}_{weather}_{date}_{original_name}``
        для предотвращения коллизий при копировании.
    """
    records: list[tuple[Path, str, str]] = []
    # Счётчик изображений по парковкам для логирования.
    lot_counts: dict[str, int] = defaultdict(int)

    for img_path in sorted(extracted_root.rglob("*")):
        # Фильтруем только файлы с допустимыми расширениями.
        if not img_path.is_file():
            continue
        if img_path.suffix.lower() not in _IMAGE_EXTENSIONS:
            continue

        # Ожидаемые части пути (относительно extracted_root):
        #   parts[-1] = filename
        #   parts[-2] = Empty | Occupied  (имя класса)
        #   parts[-3] = дата (например, 2012-09-12)
        #   parts[-4] = Cloudy | Rainy | Sunny  (погода)
        #   parts[-5] = PUCPR | UFPR04 | UFPR05  (парковка)
        try:
            rel_parts = img_path.relative_to(extracted_root).parts
        except ValueError:
            logger.warning("Не удалось вычислить относительный путь: %s", img_path)
            continue

        if len(rel_parts) < 5:
            logger.debug("Нестандартная глубина пути, пропускаем: %s", img_path)
            continue

        parking_lot = rel_parts[0]
        weather = rel_parts[1]
        date_str = rel_parts[2]
        class_folder = rel_parts[3]
        original_name = rel_parts[4]

        # Нормализуем имя класса: принимаем Empty / empty / Occupied / occupied.
        class_lower = class_folder.strip().lower()
        if class_lower == "empty":
            class_name = "Empty"
        elif class_lower == "occupied":
            class_name = "Occupied"
        else:
            logger.debug(
                "Неизвестный класс '%s', пропускаем: %s", class_folder, img_path
            )
            continue

        # Уникальное имя файла для предотвращения коллизий между парковками.
        unique_filename = f"{parking_lot}_{weather}_{date_str}_{original_name}"

        records.append((img_path, class_name, unique_filename))
        lot_counts[parking_lot] += 1

    logger.info(
        "Всего собрано изображений: %d. По парковкам: %s",
        len(records),
        dict(lot_counts),
    )
    return records


def _split_by_parking_lot(
    records: list[tuple[Path, str, str]],
    extracted_root: Path,
    seed: int,
) -> dict[str, list[tuple[Path, str, str]]]:
    """
    Разбиение по парковкам: PUCPR + UFPR04 → train/val, UFPR05 → test.

    Внутри обучающих парковок выполняется стратифицированное разбиение
    на train / val (85 / 15) для сохранения баланса классов.
    """
    train_lot_names = {lot.lower() for lot in _TRAIN_LOTS}
    test_lot_name = _TEST_LOT.lower()

    train_val_records: list[tuple[Path, str, str]] = []
    test_records: list[tuple[Path, str, str]] = []

    for rec in records:
        src_path = rec[0]
        try:
            lot_name = src_path.relative_to(extracted_root).parts[0].lower()
        except (ValueError, IndexError):
            continue

        if lot_name == test_lot_name:
            test_records.append(rec)
        elif lot_name in train_lot_names:
            train_val_records.append(rec)

    labels = [cls for _, cls, _ in train_val_records]
    train_records, val_records = train_test_split(
        train_val_records,
        test_size=_VAL_RATIO,
        stratify=labels,
        random_state=seed,
    )

    logger.info(
        "Разбиение по парковкам — train=%d (PUCPR+UFPR04 85%%) | "
        "val=%d (PUCPR+UFPR04 15%%) | test=%d (UFPR05)",
        len(train_records),
        len(val_records),
        len(test_records),
    )
    return {
        "train": train_records,
        "val": val_records,
        "test": test_records,
    }


def _log_parking_lot_distribution(
    split_records: dict[str, list[tuple[Path, str, str]]],
) -> None:
    """
    Выводит в лог количество изображений от каждой парковки в каждом разбиении.

    Параметры
    ----------
    split_records:
        Отображение имени разбиения в список кортежей
        ``(filepath, class_name, unique_filename)``.
    """
    for split, records in split_records.items():
        lot_cls_counts: dict[str, dict[str, int]] = defaultdict(
            lambda: defaultdict(int)
        )
        for src_path, class_name, _ in records:
            # Имя парковки — первая часть относительного пути.
            # unique_filename имеет вид {lot}_{weather}_{date}_{orig}.
            # Восстанавливаем имя парковки из уникального имени файла.
            lot_name = src_path.parts[-5] if len(src_path.parts) >= 5 else "unknown"
            lot_cls_counts[lot_name][class_name] += 1

        for lot, cls_counts in sorted(lot_cls_counts.items()):
            parts = ", ".join(
                f"{cls}={cnt}" for cls, cnt in sorted(cls_counts.items())
            )
            logger.info("  [%s] %s: %s", split, lot, parts)


def _copy_images(
    split_records: dict[str, list[tuple[Path, str, str]]],
    data_dir: Path,
) -> DatasetInfo:
    """
    Копирует изображения в дерево ImageFolder и возвращает статистику датасета.

    Для каждого кортежа ``(src_path, class_name, unique_filename)``
    исходное изображение копируется (через ``shutil.copy2``) по пути::

        data_dir/{разбиение}/{class_name}/{unique_filename}

    Параметры
    ----------
    split_records:
        Отображение имени разбиения в список кортежей
        ``(filepath, class_name, unique_filename)``.
    data_dir:
        Корень дерева ImageFolder (уже созданного функцией
        ``_build_empty_imagefolder_tree``).

    Возвращает
    ----------
    DatasetInfo
        Счётчики по разбиениям и классам.
    """
    counts: dict[str, int] = {"train": 0, "val": 0, "test": 0}
    distribution: dict[str, dict[str, int]] = {
        split: {cls: 0 for cls in _CLASS_NAMES}
        for split in _SPLIT_NAMES
    }

    for split in _SPLIT_NAMES:
        records = split_records.get(split, [])

        for src_path, class_name, unique_filename in records:
            dst_dir = data_dir / split / class_name
            dst_path = dst_dir / unique_filename

            if not dst_path.exists():
                shutil.copy2(src_path, dst_path)

            counts[split] += 1
            distribution[split][class_name] += 1

        logger.info(
            "Разбиение '%s' — скопировано %d изображений (%s).",
            split,
            counts[split],
            ", ".join(
                f"{cls}={distribution[split][cls]}" for cls in _CLASS_NAMES
            ),
        )

    return DatasetInfo(
        num_train=counts["train"],
        num_val=counts["val"],
        num_test=counts["test"],
        class_distribution=distribution,
    )


def _count_existing_dataset(data_dir: Path) -> DatasetInfo:
    """
    Подсчитывает изображения в уже подготовленном дереве ImageFolder
    и возвращает экземпляр ``DatasetInfo``.

    Используется, когда ``_is_already_prepared`` сообщает о завершённой
    подготовке, чтобы вернуть точную статистику без повторного запуска
    подготовки.
    """
    counts: dict[str, int] = {}
    distribution: dict[str, dict[str, int]] = {}

    for split in _SPLIT_NAMES:
        split_total = 0
        distribution[split] = {}
        for cls in _CLASS_NAMES:
            cls_dir = data_dir / split / cls
            n = len(list(cls_dir.glob("*.*"))) if cls_dir.is_dir() else 0
            distribution[split][cls] = n
            split_total += n
        counts[split] = split_total

    return DatasetInfo(
        num_train=counts["train"],
        num_val=counts["val"],
        num_test=counts["test"],
        class_distribution=distribution,
    )


# ---------------------------------------------------------------------------
# Публичное API
# ---------------------------------------------------------------------------

def download_and_prepare_dataset(data_dir: Path, seed: int = 42) -> Path:
    """
    Загружает датасет PKLot с Kaggle и организует его в структуру ImageFolder.

    Разбиение по парковкам:
    - train / val — PUCPR + UFPR04 (все погоды), стратификация 85 / 15.
    - test — UFPR05 (все погоды).
    """
    data_dir = Path(data_dir)

    # ------------------------------------------------------------------
    # Быстрый путь: подготовка уже выполнена
    # ------------------------------------------------------------------
    if _is_already_prepared(data_dir):
        logger.info(
            "Структура ImageFolder уже существует в '%s' — подготовка пропущена.",
            data_dir,
        )
        info = _count_existing_dataset(data_dir)
        info.log_summary()
        return data_dir

    logger.info("Начало подготовки датасета PKLot в '%s'.", data_dir)

    # ------------------------------------------------------------------
    # Шаг 1: Загрузка датасета с Kaggle
    # ------------------------------------------------------------------
    extracted_root = _download_kaggle(_KAGGLE_DATASET)

    # ------------------------------------------------------------------
    # Шаг 2: Сбор всех изображений со всех парковок
    # ------------------------------------------------------------------
    all_records = _collect_all_images(extracted_root)

    if not all_records:
        raise RuntimeError(
            "Не найдено ни одного изображения в загруженном датасете PKLot. "
            "Проверьте корректность загрузки и структуру архива."
        )

    # ------------------------------------------------------------------
    # Шаг 3: Разбиение по парковкам
    # ------------------------------------------------------------------
    logger.info(
        "Разбиение по парковкам: train/val=%s, test=%s (%d изображений, seed=%d).",
        _TRAIN_LOTS,
        _TEST_LOT,
        len(all_records),
        seed,
    )
    split_records = _split_by_parking_lot(all_records, extracted_root, seed=seed)

    # ------------------------------------------------------------------
    # Шаг 4: Создание дерева директорий ImageFolder
    # ------------------------------------------------------------------
    _build_empty_imagefolder_tree(data_dir)

    # ------------------------------------------------------------------
    # Шаг 5: Копирование изображений
    # ------------------------------------------------------------------
    logger.info("Копирование изображений в структуру ImageFolder …")
    info = _copy_images(split_records, data_dir)

    # ------------------------------------------------------------------
    # Шаг 6: Детальное логирование распределения по парковкам
    # ------------------------------------------------------------------
    logger.info("Распределение изображений по парковкам в каждом разбиении:")
    _log_parking_lot_distribution(split_records)
    info.log_summary()

    # ------------------------------------------------------------------
    # Шаг 7: Запись файла-сторожа
    # ------------------------------------------------------------------
    _sentinel_path(data_dir).write_text(
        f"PKLot prepared: train={info.num_train}, val={info.num_val}, test={info.num_test}\n",
        encoding="utf-8",
    )
    logger.info(
        "Подготовка завершена. Файл-сторож записан в '%s'.",
        _sentinel_path(data_dir),
    )

    return data_dir


def get_data_loaders(
    data_dir: Path,
    batch_size: int,
    train_transform: Any,
    val_transform: Any,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """
    Создаёт и возвращает экземпляры ``DataLoader`` для обучения, валидации
    и тестирования.

    Использует ``torchvision.datasets.ImageFolder``, ожидающий структуру
    директорий, созданную ``download_and_prepare_dataset``::

        data_dir/
            train/{Empty,Occupied}/
            val/{Empty,Occupied}/
            test/{Empty,Occupied}/

    Параметры
    ----------
    data_dir:
        Корень подготовленного дерева ImageFolder (значение, возвращённое
        ``download_and_prepare_dataset``).
    batch_size:
        Количество образцов в мини-батче (обычно ``Config.BATCH_SIZE = 32``).
    train_transform:
        Пайплайн аугментации для обучающих изображений (например, из
        ``get_train_transforms()``).
    val_transform:
        Детерминированный пайплайн предобработки для изображений валидации
        и тестирования (например, из ``get_val_transforms()``).

    Возвращает
    ----------
    tuple[DataLoader, DataLoader, DataLoader]
        ``(train_loader, val_loader, test_loader)``, готовые к итерации
        во время обучения и оценки модели.

    Примечания
    ----------
    * ``pin_memory`` включается только при наличии CUDA-устройства, следуя
      рекомендации PyTorch для снижения накладных расходов на CPU-only машинах.
    * Обучающий загрузчик использует ``shuffle=True``; оба загрузчика val
      и test используют ``shuffle=False``.
    * ``drop_last=False`` используется везде, чтобы каждое изображение
      оценивалось при валидации и тестировании, даже когда размер датасета
      не делится нацело на ``batch_size``.
    """
    data_dir = Path(data_dir)
    pin_memory: bool = torch.cuda.is_available()

    # ------------------------------------------------------------------
    # Обучающий датасет
    # ------------------------------------------------------------------
    train_dir = data_dir / "train"
    train_dataset = datasets.ImageFolder(
        root=str(train_dir),
        transform=train_transform,
    )
    logger.info(
        "Обучающий датасет: %d изображений | классы: %s",
        len(train_dataset),
        train_dataset.classes,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=_NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    # ------------------------------------------------------------------
    # Валидационный датасет
    # ------------------------------------------------------------------
    val_dir = data_dir / "val"
    val_dataset = datasets.ImageFolder(
        root=str(val_dir),
        transform=val_transform,
    )
    logger.info(
        "Валидационный датасет: %d изображений | классы: %s",
        len(val_dataset),
        val_dataset.classes,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=_NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    # ------------------------------------------------------------------
    # Тестовый датасет
    # ------------------------------------------------------------------
    test_dir = data_dir / "test"
    test_dataset = datasets.ImageFolder(
        root=str(test_dir),
        transform=val_transform,
    )
    logger.info(
        "Тестовый датасет: %d изображений | классы: %s",
        len(test_dataset),
        test_dataset.classes,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=_NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    logger.info(
        "DataLoader созданы — batch_size=%d | pin_memory=%s | num_workers=%d",
        batch_size,
        pin_memory,
        _NUM_WORKERS,
    )

    return train_loader, val_loader, test_loader


In [ ]:
%%writefile models.py
"""
models.py — Фабрика для всех пяти предобученных архитектур CNN / Transformer.

Поддерживаемые архитектуры
--------------------------
1. ResNet18          — torchvision.models.resnet18
2. DenseNet121       — torchvision.models.densenet121
3. EfficientNet-B0   — torchvision.models.efficientnet_b0
4. MobileNetV3-Large — torchvision.models.mobilenet_v3_large
5. ViT-B/16          — torchvision.models.vit_b_16

Каждая модель загружается с предобученными весами ImageNet через современный
API перечисления Weights (torchvision ≥ 0.13). Финальная классификационная
голова заменяется, чтобы модель выдавала ровно ``num_classes`` логитов.

Публичное API
-------------
    from models import get_model, get_model_info

    model = get_model("ResNet18", num_classes=2)
    info  = get_model_info(model)
    # {'total_params': 11_181_634, 'trainable_params': 11_181_634,
    #  'model_size_mb': 42.6}
"""

import logging
from typing import Any

import torch
import torch.nn as nn
import torchvision.models as tv_models
from torchvision.models import (
    DenseNet121_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Large_Weights,
    ResNet18_Weights,
    ViT_B_16_Weights,
)

# ---------------------------------------------------------------------------
# Логгер модуля
# ---------------------------------------------------------------------------
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Внутренний реестр — отображает каноническое имя → функция-строитель
# ---------------------------------------------------------------------------
_SUPPORTED_NAMES: tuple[str, ...] = (
    "ResNet18",
    "DenseNet121",
    "EfficientNet-B0",
    "MobileNetV3-Large",
    "ViT-B/16",
)


# ---------------------------------------------------------------------------
# Внутренние функции-строители (по одной на архитектуру)
# ---------------------------------------------------------------------------

def _build_resnet18(num_classes: int, pretrained: bool) -> nn.Module:
    """
    Создаёт ResNet-18 и заменяет полносвязную голову.

    Исходный слой ``model.fc`` является ``nn.Linear(512, 1000)``.
    Он заменяется на ``nn.Linear(512, num_classes)``.

    Параметры
    ----------
    num_classes:
        Количество выходных классов (2 для бинарной классификации парковки).
    pretrained:
        Если ``True``, загружает предобученные веса ImageNet-1K перед
        изменением головы.

    Возвращает
    ----------
    nn.Module
        ResNet-18 с изменённой классификационной головой.
    """
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model: nn.Module = tv_models.resnet18(weights=weights)
    in_features: int = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    logger.debug(
        "ResNet18 built — head: Linear(%d, %d), pretrained=%s",
        in_features,
        num_classes,
        pretrained,
    )
    return model


def _build_densenet121(num_classes: int, pretrained: bool) -> nn.Module:
    """
    Создаёт DenseNet-121 и заменяет классификационную голову.

    Исходный слой ``model.classifier`` является ``nn.Linear(1024, 1000)``.
    Он заменяется на ``nn.Linear(1024, num_classes)``.

    Параметры
    ----------
    num_classes:
        Количество выходных классов.
    pretrained:
        Если ``True``, загружает предобученные веса ImageNet-1K.

    Возвращает
    ----------
    nn.Module
        DenseNet-121 с изменённой классификационной головой.
    """
    weights = DenseNet121_Weights.DEFAULT if pretrained else None
    model: nn.Module = tv_models.densenet121(weights=weights)
    in_features: int = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    logger.debug(
        "DenseNet121 built — head: Linear(%d, %d), pretrained=%s",
        in_features,
        num_classes,
        pretrained,
    )
    return model


def _build_efficientnet_b0(num_classes: int, pretrained: bool) -> nn.Module:
    """
    Создаёт EfficientNet-B0 и заменяет линейный слой в его классификаторе.

    Классификатор является ``nn.Sequential``:
        [0] Dropout(p=0.2)
        [1] Linear(1280, 1000)   ← заменяется

    Параметры
    ----------
    num_classes:
        Количество выходных классов.
    pretrained:
        Если ``True``, загружает предобученные веса ImageNet-1K.

    Возвращает
    ----------
    nn.Module
        EfficientNet-B0 с изменённой классификационной головой.
    """
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model: nn.Module = tv_models.efficientnet_b0(weights=weights)
    in_features: int = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    logger.debug(
        "EfficientNet-B0 built — head: Linear(%d, %d), pretrained=%s",
        in_features,
        num_classes,
        pretrained,
    )
    return model


def _build_mobilenet_v3_large(num_classes: int, pretrained: bool) -> nn.Module:
    """
    Создаёт MobileNetV3-Large и заменяет финальный линейный слой.

    Классификатор является ``nn.Sequential``:
        [0] Linear(960, 1280)
        [1] Hardswish()
        [2] Dropout(p=0.2)
        [3] Linear(1280, 1000)   ← заменяется

    Параметры
    ----------
    num_classes:
        Количество выходных классов.
    pretrained:
        Если ``True``, загружает предобученные веса ImageNet-1K.

    Возвращает
    ----------
    nn.Module
        MobileNetV3-Large с изменённой классификационной головой.
    """
    weights = MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
    model: nn.Module = tv_models.mobilenet_v3_large(weights=weights)
    in_features: int = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    logger.debug(
        "MobileNetV3-Large built — head: Linear(%d, %d), pretrained=%s",
        in_features,
        num_classes,
        pretrained,
    )
    return model


def _build_vit_b_16(num_classes: int, pretrained: bool) -> nn.Module:
    """
    Создаёт Vision Transformer ViT-B/16 и заменяет классификационную голову.

    Модель предоставляет ``model.heads.head``, который является слоем
    ``nn.Linear(768, 1000)``. Он заменяется на ``nn.Linear(768, num_classes)``.

    Параметры
    ----------
    num_classes:
        Количество выходных классов.
    pretrained:
        Если ``True``, загружает предобученные веса ImageNet-1K.

    Возвращает
    ----------
    nn.Module
        ViT-B/16 с изменённой классификационной головой.
    """
    weights = ViT_B_16_Weights.DEFAULT if pretrained else None
    model: nn.Module = tv_models.vit_b_16(weights=weights)
    in_features: int = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    logger.debug(
        "ViT-B/16 built — head: Linear(%d, %d), pretrained=%s",
        in_features,
        num_classes,
        pretrained,
    )
    return model


# ---------------------------------------------------------------------------
# Внутренняя таблица диспетчеризации
# ---------------------------------------------------------------------------
_BUILDERS: dict[str, Any] = {
    "ResNet18": _build_resnet18,
    "DenseNet121": _build_densenet121,
    "EfficientNet-B0": _build_efficientnet_b0,
    "MobileNetV3-Large": _build_mobilenet_v3_large,
    "ViT-B/16": _build_vit_b_16,
}


# ---------------------------------------------------------------------------
# Публичное API
# ---------------------------------------------------------------------------

def get_model(
    model_name: str,
    num_classes: int,
    pretrained: bool = True,
) -> nn.Module:
    """
    Фабричная функция — возвращает одну из пяти поддерживаемых архитектур
    с финальной классификационной головой, адаптированной под ``num_classes``
    выходов.

    Параметры
    ----------
    model_name:
        Один из канонических идентификаторов архитектур, определённых в
        ``Config.MODEL_NAMES``:

        * ``"ResNet18"``
        * ``"DenseNet121"``
        * ``"EfficientNet-B0"``
        * ``"MobileNetV3-Large"``
        * ``"ViT-B/16"``

    num_classes:
        Количество выходных логитов. Для бинарной классификации парковки
        это 2 (``Config.NUM_CLASSES``).

    pretrained:
        Инициализировать ли базовую сеть весами ImageNet-1K.
        Устанавливается в ``False`` только для юнит-тестов или ablation-
        исследований. По умолчанию ``True``.

    Возвращает
    ----------
    nn.Module
        Экземпляр модели с изменённой головой в режиме оценки.
        Вызовите ``.train()`` перед циклом обучения при необходимости.

    Исключения
    ----------
    ValueError
        Если ``model_name`` не является одним из пяти поддерживаемых имён.

    Примеры
    --------
    >>> from models import get_model
    >>> model = get_model("ResNet18", num_classes=2)
    >>> model  # doctest: +ELLIPSIS
    ResNet(...)
    """
    if model_name not in _BUILDERS:
        supported = ", ".join(f'"{n}"' for n in _SUPPORTED_NAMES)
        raise ValueError(
            f"Unknown model name '{model_name}'. "
            f"Supported values are: {supported}."
        )

    logger.info("Building model '%s' (num_classes=%d, pretrained=%s) …",
                model_name, num_classes, pretrained)

    builder = _BUILDERS[model_name]
    model: nn.Module = builder(num_classes, pretrained)

    logger.info("Model '%s' ready.", model_name)
    return model


def get_model_info(model: nn.Module) -> dict[str, Any]:
    """
    Вычислить базовую статистику о модели PyTorch.

    Параметры
    ----------
    model:
        Любой экземпляр ``nn.Module`` (как правило, возвращённый :func:`get_model`).

    Возвращает
    -------
    dict со следующими ключами:

    ``total_params`` : int
        Общее количество скалярных параметров (обучаемых и замороженных).

    ``trainable_params`` : int
        Количество параметров, у которых ``requires_grad`` равно ``True``.

    ``model_size_mb`` : float
        Оценочный размер модели в мегабайтах при хранении в формате float32:
        ``total_params * 4 / (1024 * 1024)``.

    Примеры
    --------
    >>> from models import get_model, get_model_info
    >>> model = get_model("ResNet18", num_classes=2)
    >>> info = get_model_info(model)
    >>> info["total_params"]  # doctest: +SKIP
    11181634
    >>> isinstance(info["model_size_mb"], float)
    True
    """
    total_params: int = sum(p.numel() for p in model.parameters())
    trainable_params: int = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    # Float32 = 4 байта на параметр
    model_size_mb: float = total_params * 4 / (1024 * 1024)

    logger.debug(
        "Model info — total: %d, trainable: %d, size: %.2f MB",
        total_params,
        trainable_params,
        model_size_mb,
    )

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "model_size_mb": model_size_mb,
    }


In [ ]:
%%writefile utils.py
"""
utils.py — Утилиты обучения для классификатора занятости парковочных мест.

Данный модуль предоставляет:

* ``EarlyStopping``      — остановка обучения при прекращении улучшения потерь на валидации.
* ``ModelCheckpoint``    — автоматическое сохранение лучших весов модели на диск.
* ``compute_metrics``    — точность, precision, recall, F1 и ROC-AUC по истинным
                           меткам и предсказанным вероятностям.
* ``plot_training_curves``  — графики потерь и точности по эпохам.
* ``plot_confusion_matrix`` — тепловая карта матрицы ошибок с аннотациями.
* ``plot_roc_curve``        — ROC-кривая с аннотацией значения AUC.
* ``save_comparison_table`` — сводная таблица CSV + XLSX по всем пяти архитектурам.
* ``analyze_results``       — текстовый анализ каждой модели с итоговой
                              рекомендацией.

Использование
-----
    from utils import (
        EarlyStopping, ModelCheckpoint, compute_metrics,
        plot_training_curves, plot_confusion_matrix, plot_roc_curve,
        save_comparison_table, analyze_results,
    )
"""

import logging
import math
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# Использовать неинтерактивный бэкенд, чтобы графики можно было сохранять
# в средах без дисплея (Google Colab, серверы).
matplotlib.use("Agg")

# ---------------------------------------------------------------------------
# Логгер уровня модуля
# ---------------------------------------------------------------------------
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Ранняя остановка
# ---------------------------------------------------------------------------
class EarlyStopping:
    """
    Отслеживать валидационную метрику и сигнализировать о необходимости досрочной остановки.

    По умолчанию (``mode="max"``), обучение считается остановившимся, если
    отслеживаемая метрика (например, F1-score) не улучшается как минимум на
    ``min_delta`` в течение ``patience`` последовательных эпох.

    Параметры
    ----------
    patience:
        Количество эпох без улучшения, допустимых перед остановкой.
    min_delta:
        Минимальное абсолютное улучшение, необходимое для сброса счётчика ожидания.
        По умолчанию ``0.0``: любое строгое улучшение сбрасывает счётчик.
    mode:
        ``"max"`` — чем больше, тем лучше (F1, accuracy).
        ``"min"`` — чем меньше, тем лучше (loss).

    Атрибуты
    ----------
    best_score:
        Лучшее наблюдённое значение метрики на данный момент.
    counter:
        Текущее число последовательных эпох без улучшения.
    """

    def __init__(self, patience: int, min_delta: float = 0.0, mode: str = "max") -> None:
        if patience < 1:
            raise ValueError(f"patience must be >= 1, got {patience}")
        if mode not in ("min", "max"):
            raise ValueError(f"mode must be 'min' or 'max', got '{mode}'")
        self.patience: int = patience
        self.min_delta: float = min_delta
        self.mode: str = mode
        self.best_score: float = -math.inf if mode == "max" else math.inf
        self.counter: int = 0

    def _is_improvement(self, current: float) -> bool:
        if self.mode == "max":
            return current > self.best_score + self.min_delta
        return current < self.best_score - self.min_delta

    # ------------------------------------------------------------------
    def __call__(self, metric_value: float) -> bool:
        """
        Оценить последнее значение метрики и решить, следует ли останавливаться.

        Параметры
        ----------
        metric_value:
            Валидационная метрика, измеренная в конце текущей эпохи
            (например, F1-score при ``mode="max"``).

        Возвращает
        -------
        bool
            ``True``, когда лимит ожидания исчерпан и обучение следует
            прекратить; ``False`` в противном случае.
        """
        if self._is_improvement(metric_value):
            logger.debug(
                "EarlyStopping: metric improved %.6f → %.6f",
                self.best_score,
                metric_value,
            )
            self.best_score = metric_value
            self.counter = 0
            return False

        self.counter += 1
        logger.debug(
            "EarlyStopping: no improvement (%.6f vs best %.6f), counter=%d/%d",
            metric_value,
            self.best_score,
            self.counter,
            self.patience,
        )
        if self.counter >= self.patience:
            logger.info(
                "EarlyStopping triggered after %d epochs with no improvement.",
                self.patience,
            )
            return True
        return False

    # ------------------------------------------------------------------
    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"patience={self.patience}, "
            f"min_delta={self.min_delta}, "
            f"mode='{self.mode}', "
            f"best_score={self.best_score:.6f}, "
            f"counter={self.counter})"
        )


# ---------------------------------------------------------------------------
# Сохранение контрольных точек модели
# ---------------------------------------------------------------------------
class ModelCheckpoint:
    """
    Сохранять веса модели каждый раз, когда отслеживаемая метрика достигает нового максимума.

    По умолчанию (``mode="max"``), сохранение происходит, когда метрика (например, F1-score)
    превышает предыдущий лучший результат.

    Параметры
    ----------
    save_path:
        Полный путь файловой системы (включая имя файла), куда будет записан файл ``.pth``.
        Родительские директории создаются автоматически.
    mode:
        ``"max"`` — чем больше, тем лучше (F1, accuracy).
        ``"min"`` — чем меньше, тем лучше (loss).

    Атрибуты
    ----------
    best_value:
        Лучшее значение метрики, для которого был сохранён чекпоинт.
    """

    def __init__(self, save_path: Path, mode: str = "max") -> None:
        if mode not in ("min", "max"):
            raise ValueError(f"mode must be 'min' or 'max', got '{mode}'")
        self.save_path: Path = Path(save_path)
        self.save_path.parent.mkdir(parents=True, exist_ok=True)
        self.mode: str = mode
        self.best_value: float = -math.inf if mode == "max" else math.inf

    def _is_improvement(self, current: float) -> bool:
        if self.mode == "max":
            return current > self.best_value
        return current < self.best_value

    # ------------------------------------------------------------------
    def __call__(self, metric_value: float, model: nn.Module) -> None:
        """
        Сохранить веса модели, если метрика улучшилась.

        Параметры
        ----------
        metric_value:
            Валидационная метрика (например, F1-score) в конце текущей эпохи.
        model:
            Модель PyTorch, state dict которой следует сохранить.
        """
        if self._is_improvement(metric_value):
            logger.info(
                "ModelCheckpoint: metric improved %.6f → %.6f — saving to %s",
                self.best_value,
                metric_value,
                self.save_path,
            )
            self.best_value = metric_value
            torch.save(model.state_dict(), self.save_path)
        else:
            logger.debug(
                "ModelCheckpoint: metric %.6f did not improve best %.6f — skip",
                metric_value,
                self.best_value,
            )

    # ------------------------------------------------------------------
    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"save_path={self.save_path!r}, "
            f"mode='{self.mode}', "
            f"best_value={self.best_value:.6f})"
        )


# ---------------------------------------------------------------------------
# Метрики
# ---------------------------------------------------------------------------
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> dict[str, float]:
    """
    Вычислить метрики бинарной классификации по истинным меткам и предсказаниям.

    Параметры
    ----------
    y_true:
        Одномерный целочисленный массив истинных индексов классов (0 = Empty,
        1 = Occupied).
    y_pred:
        Одномерный целочисленный массив предсказанных индексов классов (argmax логитов).
    y_proba:
        Одномерный массив вещественных чисел — предсказанные вероятности для
        *положительного* класса (индекс 1 — Occupied). Используется исключительно
        для вычисления ROC-AUC.

    Возвращает
    -------
    dict[str, float]
        Словарь со следующими ключами:

        * ``accuracy``  — доля правильно классифицированных образцов.
        * ``precision`` — взвешенная точность по обоим классам.
        * ``recall``    — взвешенная полнота по обоим классам.
        * ``f1``        — взвешенный F1-score.
        * ``roc_auc``   — площадь под ROC-кривой.

    Исключения
    ------
    ValueError
        Если входные массивы имеют несовпадающую длину.
    """
    if len(y_true) != len(y_pred) or len(y_true) != len(y_proba):
        raise ValueError(
            "y_true, y_pred, and y_proba must all have the same length; "
            f"got {len(y_true)}, {len(y_pred)}, {len(y_proba)}"
        )

    accuracy: float = float(accuracy_score(y_true, y_pred))
    precision: float = float(
        precision_score(y_true, y_pred, average="weighted", zero_division=0)
    )
    recall: float = float(
        recall_score(y_true, y_pred, average="weighted", zero_division=0)
    )
    f1: float = float(
        f1_score(y_true, y_pred, average="weighted", zero_division=0)
    )
    roc_auc: float = float(roc_auc_score(y_true, y_proba))

    metrics: dict[str, float] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

    logger.info(
        "Metrics — acc=%.4f | prec=%.4f | rec=%.4f | f1=%.4f | auc=%.4f",
        accuracy,
        precision,
        recall,
        f1,
        roc_auc,
    )
    return metrics


# ---------------------------------------------------------------------------
# Вспомогательные функции построения графиков
# ---------------------------------------------------------------------------
def plot_training_curves(
    history: dict[str, list[float]],
    model_name: str,
    save_dir: Path,
) -> None:
    """
    Построить и сохранить графики кривых обучения для одной модели.

    Записываются два отдельных файла PNG:

    * ``{model_name}_loss.png``     — потери на обучении и валидации по эпохам.
    * ``{model_name}_accuracy.png`` — точность на обучении и валидации по эпохам.

    Параметры
    ----------
    history:
        Словарь, сформированный в процессе обучения, с ключами ``train_loss``,
        ``val_loss``, ``train_acc`` и ``val_acc``, каждый из которых содержит
        список скалярных значений — по одному на эпоху.
    model_name:
        Читаемый идентификатор модели, используемый в заголовке графика и
        имени файла (например, ``"ResNet18"``).
    save_dir:
        Директория, в которую будут сохранены файлы PNG. Создаётся автоматически,
        если не существует.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    epochs: list[int] = list(range(1, len(history["train_loss"]) + 1))

    # ── График потерь ────────────────────────────────────────────────────────
    fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
    ax_loss.plot(epochs, history["train_loss"], label="Train Loss", linewidth=2)
    ax_loss.plot(epochs, history["val_loss"], label="Val Loss", linewidth=2)
    ax_loss.set_title(f"{model_name} — Training & Validation Loss", fontsize=14)
    ax_loss.set_xlabel("Epoch", fontsize=12)
    ax_loss.set_ylabel("Loss", fontsize=12)
    ax_loss.legend(fontsize=11)
    ax_loss.grid(True, linestyle="--", alpha=0.6)
    fig_loss.tight_layout()
    loss_path: Path = save_dir / f"{model_name}_loss.png"
    fig_loss.savefig(loss_path, dpi=150, bbox_inches="tight")
    plt.close(fig_loss)
    logger.info("Saved loss curve → %s", loss_path)

    # ── График точности ────────────────────────────────────────────────────────
    fig_acc, ax_acc = plt.subplots(figsize=(8, 5))
    ax_acc.plot(epochs, history["train_acc"], label="Train Accuracy", linewidth=2)
    ax_acc.plot(epochs, history["val_acc"], label="Val Accuracy", linewidth=2)
    ax_acc.set_title(f"{model_name} — Training & Validation Accuracy", fontsize=14)
    ax_acc.set_xlabel("Epoch", fontsize=12)
    ax_acc.set_ylabel("Accuracy", fontsize=12)
    ax_acc.legend(fontsize=11)
    ax_acc.grid(True, linestyle="--", alpha=0.6)
    fig_acc.tight_layout()
    acc_path: Path = save_dir / f"{model_name}_accuracy.png"
    fig_acc.savefig(acc_path, dpi=150, bbox_inches="tight")
    plt.close(fig_acc)
    logger.info("Saved accuracy curve → %s", acc_path)


# ---------------------------------------------------------------------------
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: tuple[str, ...] | list[str],
    model_name: str,
    save_dir: Path,
) -> None:
    """
    Построить и сохранить аннотированную матрицу ошибок для одной модели.

    Цветовая карта варьируется от белого (нулевые значения) до насыщенного синего
    (максимальное значение). Каждая ячейка аннотирована целочисленным значением.

    Параметры
    ----------
    y_true:
        Одномерный целочисленный массив истинных индексов классов.
    y_pred:
        Одномерный целочисленный массив предсказанных индексов классов.
    class_names:
        Упорядоченная последовательность меток классов, соответствующая порядку
        индексов в ``y_true`` и ``y_pred`` (например, ``("Empty", "Occupied")``).
    model_name:
        Читаемый идентификатор модели, используемый в заголовке и имени файла.
    save_dir:
        Директория, в которую будет сохранён файл PNG. Создаётся автоматически,
        если не существует.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    cm: np.ndarray = confusion_matrix(y_true, y_pred)
    num_classes: int = len(class_names)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)

    ax.set_title(f"{model_name} — Confusion Matrix", fontsize=14)
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)

    tick_marks: list[int] = list(range(num_classes))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, fontsize=11)
    ax.set_yticklabels(class_names, fontsize=11)

    # Аннотировать каждую ячейку её значением; использовать белый текст на тёмных ячейках.
    thresh: float = cm.max() / 2.0
    for row in range(num_classes):
        for col in range(num_classes):
            text_color: str = "white" if cm[row, col] > thresh else "black"
            ax.text(
                col,
                row,
                str(cm[row, col]),
                ha="center",
                va="center",
                color=text_color,
                fontsize=13,
                fontweight="bold",
            )

    fig.tight_layout()
    cm_path: Path = save_dir / f"{model_name}_confusion_matrix.png"
    fig.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved confusion matrix → %s", cm_path)


# ---------------------------------------------------------------------------
def plot_roc_curve(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    model_name: str,
    save_dir: Path,
) -> None:
    """
    Построить и сохранить ROC-кривую со значением AUC в легенде.

    Диагональная референсная линия (базовый уровень случайного классификатора)
    отображается серым цветом.

    Параметры
    ----------
    y_true:
        Одномерный целочисленный массив истинных индексов классов.
    y_proba:
        Одномерный массив вещественных чисел — предсказанные вероятности для
        положительного класса (индекс 1 — Occupied).
    model_name:
        Читаемый идентификатор модели, используемый в заголовке и имени файла.
    save_dir:
        Директория, в которую будет сохранён файл PNG. Создаётся автоматически,
        если не существует.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    fpr: np.ndarray
    tpr: np.ndarray
    _thresholds: np.ndarray
    fpr, tpr, _thresholds = roc_curve(y_true, y_proba)
    roc_auc_value: float = float(auc(fpr, tpr))

    fig, ax = plt.subplots(figsize=(7, 5))

    # Диагональная референсная линия (случайный классификатор).
    ax.plot(
        [0.0, 1.0],
        [0.0, 1.0],
        color="grey",
        linestyle="--",
        linewidth=1.5,
        label="Random classifier (AUC = 0.50)",
    )

    # ROC-кривая данной модели.
    ax.plot(
        fpr,
        tpr,
        color="steelblue",
        linewidth=2.5,
        label=f"{model_name} (AUC = {roc_auc_value:.4f})",
    )

    ax.set_title(f"{model_name} — ROC Curve", fontsize=14)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.legend(fontsize=11, loc="lower right")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])

    fig.tight_layout()
    roc_path: Path = save_dir / f"{model_name}_roc_curve.png"
    fig.savefig(roc_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved ROC curve → %s", roc_path)


# ---------------------------------------------------------------------------
# Сравнительная таблица
# ---------------------------------------------------------------------------
def save_comparison_table(
    results: list[dict[str, Any]],
    save_dir: Path,
) -> None:
    """
    Собрать, аннотировать и сохранить сравнительную таблицу по всем архитектурам.

    Таблица сортируется по F1-score в порядке убывания. Добавляется столбец ``Best``,
    значение которого равно ``"*"`` для лучшей модели и пусто для всех остальных.

    В директории ``save_dir`` записываются два файла:

    * ``comparison.csv``  — разделённый запятыми, UTF-8.
    * ``comparison.xlsx`` — книга Excel с автоматически подобранной шириной столбцов
      и строкой лучшей модели, выделенной светло-зелёным цветом.

    Параметры
    ----------
    results:
        Список словарей — по одному на архитектуру — каждый из которых содержит
        следующие ключи (все должны присутствовать):

        * ``Architecture``   — строка с названием модели.
        * ``Accuracy``       — вещественное число в [0, 1].
        * ``Precision``      — вещественное число в [0, 1].
        * ``Recall``         — вещественное число в [0, 1].
        * ``F1``             — вещественное число в [0, 1].
        * ``ROC_AUC``        — вещественное число в [0, 1].
        * ``Inference_Time`` — секунды на батч (вещественное число).
        * ``Parameters``     — общее количество обучаемых параметров (int).
        * ``Model_Size``     — размер чекпоинта в мегабайтах (вещественное число).
        * ``Epochs``         — фактически пройденное число эпох (int).
        * ``Training_Time``  — общее астрономическое время обучения в секундах (вещественное число).

    save_dir:
        Директория, в которую будут записаны выходные файлы. Создаётся
        автоматически, если не существует.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Определить канонический порядок столбцов.
    columns: list[str] = [
        "Architecture",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC",
        "Training Time",
        "Inference Time (ms/image)",
        "FPS",
        "Model Size (MB)",
        "Number of Parameters",
        "Epochs",
    ]

    df: pd.DataFrame = pd.DataFrame(results, columns=columns)
    df.sort_values(by="F1", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Отметить лучшую модель (после сортировки первая строка — наивысший F1).
    df.insert(0, "Best", "")
    df.at[0, "Best"] = "*"

    # ── CSV ──────────────────────────────────────────────────────────────────
    csv_path: Path = save_dir / "comparison.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    logger.info("Saved comparison CSV → %s", csv_path)

    # ── XLSX ─────────────────────────────────────────────────────────────────
    xlsx_path: Path = save_dir / "comparison.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Comparison")

        workbook = writer.book
        worksheet = writer.sheets["Comparison"]

        # Автоматически подобрать ширину столбцов.
        for col_cells in worksheet.columns:
            max_length: int = 0
            col_letter: str = col_cells[0].column_letter
            for cell in col_cells:
                if cell.value is not None:
                    cell_str_len: int = len(str(cell.value))
                    if cell_str_len > max_length:
                        max_length = cell_str_len
            adjusted_width: float = max_length + 4
            worksheet.column_dimensions[col_letter].width = adjusted_width

        # Выделить строку лучшей модели (строка 2 — строка 1 является заголовком).
        from openpyxl.styles import PatternFill

        green_fill = PatternFill(
            start_color="C6EFCE",
            end_color="C6EFCE",
            fill_type="solid",
        )
        num_cols: int = len(df.columns)
        for col_idx in range(1, num_cols + 1):
            worksheet.cell(row=2, column=col_idx).fill = green_fill

    logger.info("Saved comparison XLSX → %s", xlsx_path)


# ---------------------------------------------------------------------------
# Текстовый анализ
# ---------------------------------------------------------------------------

# Текстовые описания по архитектурам, заполняемые фактическими значениями метрик
# из списка результатов во время вызова.
_ARCH_PROFILES: dict[str, dict[str, str]] = {
    "ResNet18": {
        "strengths": (
            "Очень быстрое обучение и инференс благодаря неглубокой остаточной "
            "сети (18 слоёв). Малое количество параметров упрощает развёртывание "
            "и дообучение на ограниченном оборудовании. Остаточные связи устраняют "
            "проблему затухающих градиентов и обеспечивают стабильное обучение с первой эпохи."
        ),
        "weaknesses": (
            "Меньшая ёмкость по сравнению с более глубокими или широкими сетями; "
            "может выйти на плато с более низким потолком точности для сложных "
            "визуальных паттернов. Хуже представляет тонкие вариации освещённости "
            "или окклюзии по сравнению с архитектурами EfficientNet или ViT."
        ),
        "speed_note": (
            "Самая быстрая архитектура в сравнении. Подходит для инференса "
            "в реальном времени на граничных устройствах или при ограниченных ресурсах."
        ),
    },
    "DenseNet121": {
        "strengths": (
            "Плотная связность стимулирует повторное использование признаков между "
            "слоями, что может улучшить обобщение при ограниченных данных. Сильный "
            "поток градиентов через сеть снижает переобучение на относительно "
            "небольших датасетах, таких как PKLot."
        ),
        "weaknesses": (
            "Больший объём используемой памяти при обучении по сравнению с ResNet18, "
            "так как активации всех предыдущих слоёв должны конкатенироваться. "
            "Инференс медленнее, чем у лёгких моделей вроде MobileNetV3."
        ),
        "speed_note": (
            "Умеренная скорость. Потребление памяти при обучении выше среднего; "
            "задержка инференса приемлема для пакетной обработки."
        ),
    },
    "EfficientNet-B0": {
        "strengths": (
            "Благодаря составному масштабированию глубины, ширины и разрешения "
            "EfficientNet-B0 обеспечивает высокую точность при небольшом бюджете "
            "параметров. Глубинно-разделимые свёртки делают её вычислительно "
            "эффективной при сохранении высокой представительной мощи."
        ),
        "weaknesses": (
            "Несколько более сложная динамика обучения по сравнению с обычными "
            "свёрточными сетями; расписание скорости обучения может быть более "
            "чувствительным. Требует тщательной аугментации данных во избежание "
            "переобучения на небольших датасетах."
        ),
        "speed_note": (
            "Хороший баланс точности и скорости. Одна из лучших архитектур "
            "по соотношению точности к числу FLOP в данном сравнении."
        ),
    },
    "MobileNetV3-Large": {
        "strengths": (
            "Специально разработана для инференса на устройстве: инвертированные "
            "остатки и активации hard-swish минимизируют задержку. Наименьший "
            "размер модели и наибыстрейший инференс в группе, что делает её "
            "идеальной для встраиваемых или мобильных развёртываний."
        ),
        "weaknesses": (
            "Более низкая представительная ёмкость по сравнению с EfficientNet "
            "или DenseNet для тонкозернистой классификации. Может потребоваться "
            "больше аугментаций или более длительное обучение для достижения "
            "результатов более глубоких архитектур."
        ),
        "speed_note": (
            "Самый быстрый или совместно самый быстрый инференс. Рекомендуется "
            "при строгих ограничениях на задержку или размер модели."
        ),
    },
    "ViT-B/16": {
        "strengths": (
            "Vision Transformer улавливает дальнодействующие пространственные "
            "зависимости через многоголовое самовнимание, что может быть "
            "преимуществом, когда признаки занятости парковки распределены по "
            "изображению (например, кузов автомобиля виден только в одном углу). "
            "Предобучение на больших корпусах (ImageNet-21k) переносит богатые "
            "представления признаков."
        ),
        "weaknesses": (
            "Требует наибольшего количества параметров и наибольшего времени "
            "обучения в сравнении. Архитектуры трансформеров, как правило, "
            "нуждаются в больших датасетах для раскрытия полного потенциала; "
            "при работе только с PKLot перенос обучения снижает, но не устраняет "
            "этот риск. Инференс медленнее, чем у всех CNN-аналогов."
        ),
        "speed_note": (
            "Самая медленная архитектура в сравнении. Лучше всего подходит "
            "для офлайн пакетного инференса, где точность важнее задержки."
        ),
    },
}

# Резервный профиль для архитектур, не перечисленных выше.
_DEFAULT_PROFILE: dict[str, str] = {
    "strengths": "Перенос обучения с весов ImageNet обеспечивает сильную базу признаков.",
    "weaknesses": "Профиль недоступен; оценивайте эмпирически по таблице метрик.",
    "speed_note": "Характеристики скорости не профилированы; измерьте по столбцу Inference_Time.",
}


def analyze_results(results: list[dict[str, Any]]) -> str:
    """
    Сформировать исчерпывающий текстовый анализ всех обученных моделей.

    Для каждой архитектуры анализ включает:

    * **Сильные стороны** — архитектурные преимущества.
    * **Слабые стороны** — известные ограничения.
    * **Скорость** — качественные и количественные замечания на основе фактических
      измерений ``Inference_Time`` и ``Training_Time``.
    * **Точность** — количественная сводка по фактическим метрикам.
    * **Вывод** — итоговый вердикт по каждой модели.

    После анализа по каждой модели итоговый раздел выбирает лучшую модель по
    наибольшему F1-score и объясняет логику выбора.

    Параметры
    ----------
    results:
        Список словарей с той же схемой, что принимает :func:`save_comparison_table`.

    Возвращает
    -------
    str
        Многосекционный текст анализа, готовый для вывода, логирования или
        встраивания в ячейку Markdown Jupyter-ноутбука.
    """
    if not results:
        return "No results to analyse."

    # Сортировать по F1 в порядке убывания для определения рейтинга.
    sorted_results: list[dict[str, Any]] = sorted(
        results, key=lambda r: r.get("F1", 0.0), reverse=True
    )
    best: dict[str, Any] = sorted_results[0]
    best_arch: str = best.get("Architecture", "Unknown")

    lines: list[str] = []

    lines.append("=" * 72)
    lines.append("  MODEL ANALYSIS — PARKING SPOT OCCUPANCY CLASSIFICATION")
    lines.append("=" * 72)
    lines.append("")

    # ── Блоки по каждой модели ────────────────────────────────────────────────
    for rank, res in enumerate(sorted_results, start=1):
        arch: str = res.get("Architecture", "Unknown")
        acc: float = res.get("Accuracy", 0.0)
        prec: float = res.get("Precision", 0.0)
        rec: float = res.get("Recall", 0.0)
        f1: float = res.get("F1", 0.0)
        roc: float = res.get("ROC AUC", 0.0)
        inf_time_ms: float = res.get("Inference Time (ms/image)", 0.0)
        fps: float = res.get("FPS", 0.0)
        params: int = res.get("Number of Parameters", 0)
        model_mb: float = res.get("Model Size (MB)", 0.0)
        epochs: int = res.get("Epochs", 0)
        train_time: float = res.get("Training Time", 0.0)

        profile: dict[str, str] = _ARCH_PROFILES.get(arch, _DEFAULT_PROFILE)

        header: str = f"  #{rank}  {arch}"
        if arch == best_arch:
            header += "  [BEST MODEL]"
        lines.append("-" * 72)
        lines.append(header)
        lines.append("-" * 72)

        lines.append("")
        lines.append("  Metrics")
        lines.append(f"    Accuracy      : {acc:.4f}  ({acc * 100:.2f} %)")
        lines.append(f"    Precision     : {prec:.4f}  ({prec * 100:.2f} %)")
        lines.append(f"    Recall        : {rec:.4f}  ({rec * 100:.2f} %)")
        lines.append(f"    F1 Score      : {f1:.4f}  ({f1 * 100:.2f} %)")
        lines.append(f"    ROC-AUC       : {roc:.4f}")
        lines.append("")
        lines.append("  Runtime")
        lines.append(
            f"    Epochs trained: {epochs}"
        )
        lines.append(
            f"    Training time : {train_time:.1f} s  "
            f"({train_time / 60:.1f} min)"
        )
        lines.append(
            f"    Inference time: {inf_time_ms:.2f} ms / image"
        )
        lines.append(
            f"    FPS           : {fps:.1f}"
        )
        lines.append(
            f"    Parameters    : {params:,}"
        )
        lines.append(
            f"    Model size    : {model_mb:.2f} MB"
        )
        lines.append("")
        lines.append("  Strengths")
        for sentence in _wrap_text(profile["strengths"], width=68, indent="    "):
            lines.append(sentence)
        lines.append("")
        lines.append("  Weaknesses")
        for sentence in _wrap_text(profile["weaknesses"], width=68, indent="    "):
            lines.append(sentence)
        lines.append("")
        lines.append("  Speed")
        for sentence in _wrap_text(profile["speed_note"], width=68, indent="    "):
            lines.append(sentence)
        lines.append("")

        # Вердикт по каждой модели.
        if f1 >= 0.98:
            verdict = "Outstanding classification performance. Highly recommended."
        elif f1 >= 0.95:
            verdict = "Excellent performance. Suitable for production use."
        elif f1 >= 0.90:
            verdict = "Good performance. May benefit from additional fine-tuning."
        elif f1 >= 0.80:
            verdict = "Acceptable performance. Further hyperparameter search advised."
        else:
            verdict = "Below expectations. Consider deeper fine-tuning or more data."

        lines.append("  Conclusion")
        lines.append(f"    {verdict}")
        lines.append("")

    # ── Рекомендация лучшей модели ────────────────────────────────────────────
    lines.append("=" * 72)
    lines.append("  BEST MODEL RECOMMENDATION")
    lines.append("=" * 72)
    lines.append("")
    lines.append(f"  Selected architecture : {best_arch}")
    lines.append(f"  F1 Score              : {best.get('F1', 0.0):.4f}")
    lines.append(f"  Accuracy              : {best.get('Accuracy', 0.0):.4f}")
    lines.append(f"  ROC-AUC               : {best.get('ROC AUC', 0.0):.4f}")
    lines.append("")

    rationale_lines: list[str] = [
        f"{best_arch} was selected as the best model because it achieved the "
        "highest F1 score on the hold-out test set.  F1 is the primary "
        "selection criterion because it harmonises precision and recall, "
        "which is important for parking-occupancy detection: a false "
        "negative (reporting an occupied space as empty) wastes drivers' "
        "time, while a false positive (reporting an empty space as occupied) "
        "can misdirect navigation.  Maximising F1 ensures both error types "
        "are kept in check simultaneously.",
    ]
    best_profile: dict[str, str] = _ARCH_PROFILES.get(best_arch, _DEFAULT_PROFILE)
    rationale_lines.append(
        f"From an architectural standpoint, {best_arch} benefits from: "
        + best_profile["strengths"]
    )
    rationale_lines.append(
        "Given these properties and the measured metrics, "
        f"{best_arch} is recommended for deployment in the parking-space "
        "occupancy classifier system."
    )

    for para in rationale_lines:
        for wrapped_line in _wrap_text(para, width=68, indent="  "):
            lines.append(wrapped_line)
        lines.append("")

    lines.append("=" * 72)

    analysis_text: str = "\n".join(lines)
    logger.info("Analysis generated for %d models. Best: %s", len(results), best_arch)
    return analysis_text


# ---------------------------------------------------------------------------
# Внутренняя вспомогательная функция
# ---------------------------------------------------------------------------
def _wrap_text(text: str, width: int, indent: str = "") -> list[str]:
    """
    Перенести ``text`` по ``width`` символов на строку, добавляя префикс ``indent``.

    Параметры
    ----------
    text:
        Исходный текст для переноса.
    width:
        Максимальная ширина каждой выходной строки в символах (без учёта отступа).
    indent:
        Строка, добавляемая в начало каждой выходной строки.

    Возвращает
    -------
    list[str]
        Список перенесённых строк с применённым отступом.
    """
    import textwrap

    wrapped: list[str] = textwrap.wrap(text, width=width)
    return [indent + line for line in wrapped] if wrapped else [indent + text]


In [ ]:
%%writefile train.py
"""
train.py — Полный конвейер обучения классификатора занятости парковочных мест.

Данный модуль организует сквозное обучение и оценку для всех пяти
поддерживаемых архитектур (ResNet18, DenseNet121, EfficientNet-B0,
MobileNetV3-Large, ViT-B/16).

Публичный API
-------------
- ``train_one_epoch(model, dataloader, criterion, optimizer, device)``
      Выполняет один прямой/обратный проход по всем батчам; возвращает (loss, accuracy).
- ``evaluate(model, dataloader, criterion, device)``
      Оценивает модель на DataLoader; возвращает loss, accuracy и сырые массивы.
- ``train_model(model_name, train_loader, val_loader, test_loader, config)``
      Полный конвейер обучения для одной архитектуры; возвращает словарь результатов.
- ``run_full_pipeline(config)``
      Последовательно обучает все пять моделей, сохраняет таблицы и текст анализа.

Использование
-------------
    python train.py                   # обучает все модели с конфигурацией по умолчанию
    from train import run_full_pipeline
    from config import Config
    results = run_full_pipeline(Config)
"""

from __future__ import annotations

import json
import logging
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from config import Config, get_train_transforms, get_val_transforms, seed_everything
from dataset import download_and_prepare_dataset, get_data_loaders
from models import get_model, get_model_info
from utils import (
    EarlyStopping,
    ModelCheckpoint,
    analyze_results,
    compute_metrics,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_training_curves,
    save_comparison_table,
)

# ---------------------------------------------------------------------------
# Логгер уровня модуля
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# train_one_epoch
# ---------------------------------------------------------------------------

def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """
    Выполняет одну полную эпоху обучения по всем батчам в ``dataloader``.

    Перед началом итерации модель переводится в режим обучения. Для каждого
    мини-батча прямой проход вычисляет функцию потерь, обратный проход вычисляет
    градиенты, и оптимизатор обновляет веса.

    Во время итерации отображается прогресс-бар ``tqdm``, показывающий
    скользящее среднее потерь и точности за текущую эпоху.

    Parameters
    ----------
    model:
        Модель PyTorch для обучения. Должна быть уже перемещена на ``device``.
    dataloader:
        DataLoader, возвращающий мини-батчи ``(images, labels)`` из
        обучающей выборки.
    criterion:
        Функция потерь (``nn.CrossEntropyLoss``).
    optimizer:
        Градиентный оптимизатор (``torch.optim.Adam``).
    device:
        Вычислительное устройство (``torch.device("cuda")``, ``"mps"`` или ``"cpu"``).

    Returns
    -------
    tuple[float, float]
        ``(avg_loss, accuracy)`` — среднее значение кросс-энтропийных потерь и доля
        правильно классифицированных образцов за всю эпоху.
    """
    model.train()

    running_loss: float = 0.0
    correct: int = 0
    total: int = 0

    progress_bar = tqdm(
        dataloader,
        desc="  Train",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
    )

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Обнуляем градиенты перед прямым проходом.
        optimizer.zero_grad()

        # Прямой проход.
        logits: torch.Tensor = model(images)
        loss: torch.Tensor = criterion(logits, labels)

        # Обратный проход и обновление весов.
        loss.backward()
        optimizer.step()

        # Накапливаем статистику.
        batch_size: int = labels.size(0)
        running_loss += loss.item() * batch_size
        predicted: torch.Tensor = logits.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

        # Обновляем постфикс прогресс-бара скользящими средними.
        current_loss = running_loss / total
        current_acc = correct / total
        progress_bar.set_postfix(
            loss=f"{current_loss:.4f}",
            acc=f"{current_acc:.4f}",
        )

    progress_bar.close()

    avg_loss: float = running_loss / total if total > 0 else 0.0
    accuracy: float = correct / total if total > 0 else 0.0

    return avg_loss, accuracy


# ---------------------------------------------------------------------------
# evaluate
# ---------------------------------------------------------------------------

def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, np.ndarray, np.ndarray, np.ndarray]:
    """
    Оценивает модель на всех батчах из ``dataloader`` без обновления весов.

    Модель переводится в режим оценки, а все вычисления градиентов отключаются
    через ``torch.no_grad()``. Предсказания, истинные метки и вероятности
    класса 1 собираются по всем батчам.

    Parameters
    ----------
    model:
        Модель PyTorch для оценки. Должна быть уже перемещена на ``device``.
    dataloader:
        DataLoader, возвращающий мини-батчи ``(images, labels)``. Обычно
        DataLoader валидационной или тестовой выборки.
    criterion:
        Функция потерь (``nn.CrossEntropyLoss``), используемая для вычисления
        средних потерь по всем образцам.
    device:
        Вычислительное устройство.

    Returns
    -------
    tuple[float, float, np.ndarray, np.ndarray, np.ndarray]
        - ``avg_loss``  : float — среднее значение кросс-энтропийных потерь.
        - ``accuracy``  : float — доля правильно классифицированных образцов.
        - ``y_true``    : одномерный массив numpy int32 с истинными индексами классов.
        - ``y_pred``    : одномерный массив numpy int32 с предсказанными индексами классов.
        - ``y_proba``   : одномерный массив numpy float32 с вероятностью для класса 1
                          (Occupied), полученной через ``F.softmax``.
    """
    model.eval()

    running_loss: float = 0.0
    correct: int = 0
    total: int = 0

    all_true: list[np.ndarray] = []
    all_pred: list[np.ndarray] = []
    all_proba: list[np.ndarray] = []

    with torch.no_grad():
        progress_bar = tqdm(
            dataloader,
            desc="  Eval ",
            unit="batch",
            leave=False,
            dynamic_ncols=True,
        )

        for images, labels in progress_bar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits: torch.Tensor = model(images)
            loss: torch.Tensor = criterion(logits, labels)

            batch_size: int = labels.size(0)
            running_loss += loss.item() * batch_size
            predicted: torch.Tensor = logits.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += batch_size

            # Вычисляем вероятности softmax; оставляем только вероятность класса 1 (Occupied).
            probabilities: torch.Tensor = F.softmax(logits, dim=1)
            class1_proba: torch.Tensor = probabilities[:, 1]

            all_true.append(labels.cpu().numpy().astype(np.int32))
            all_pred.append(predicted.cpu().numpy().astype(np.int32))
            all_proba.append(class1_proba.cpu().numpy().astype(np.float32))

        progress_bar.close()

    avg_loss: float = running_loss / total if total > 0 else 0.0
    accuracy: float = correct / total if total > 0 else 0.0

    y_true: np.ndarray = np.concatenate(all_true, axis=0)
    y_pred: np.ndarray = np.concatenate(all_pred, axis=0)
    y_proba: np.ndarray = np.concatenate(all_proba, axis=0)

    return avg_loss, accuracy, y_true, y_pred, y_proba


# ---------------------------------------------------------------------------
# train_model
# ---------------------------------------------------------------------------

def train_model(
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    config: type[Config],
    output_dir: Path | None = None,
) -> dict[str, Any]:
    """
    Полный конвейер обучения для одной архитектуры.

    Выполняемые шаги
    ----------------
    1. Строит модель через ``get_model()`` и перемещает её на ``config.DEVICE``.
    2. Создаёт оптимизатор ``Adam`` и функцию потерь ``CrossEntropyLoss``.
    3. Подключает колбэки ``EarlyStopping`` и ``ModelCheckpoint``.
    4. Цикл обучения: для каждой эпохи вызывает ``train_one_epoch()``, затем
       ``evaluate()`` на валидационной выборке; передаёт val loss в оба
       колбэка; досрочно завершает, если срабатывает ``EarlyStopping``.
    5. Загружает лучший чекпоинт, сохранённый ``ModelCheckpoint``.
    6. Запускает ``evaluate()`` на тестовой выборке.
    7. Вычисляет все метрики классификации через ``compute_metrics()``.
    8. Измеряет время инференса на одно изображение, усреднённое по 100 одиночным
       прямым проходам на вычислительном устройстве.
    9. Получает размер модели и количество параметров через ``get_model_info()``.
    10. Строит графики кривых обучения, матрицы ошибок и ROC-кривой.
    11. Возвращает словарь результатов со всеми ключами, необходимыми для
        ``save_comparison_table()``.

    Parameters
    ----------
    model_name:
        Одно из канонических имён из ``Config.MODEL_NAMES``
        (например, ``"ResNet18"``).
    train_loader:
        DataLoader для обучающей выборки.
    val_loader:
        DataLoader для валидационной выборки.
    test_loader:
        DataLoader для тестовой выборки.
    config:
        Класс ``Config`` (не экземпляр), предоставляющий все гиперпараметры
        и пути к файловой системе.

    Returns
    -------
    dict[str, Any]
        Словарь со следующими ключами, совместимый с
        ``save_comparison_table()`` и ``analyze_results()``:

        - ``Architecture``   : str
        - ``Accuracy``       : float
        - ``Precision``      : float
        - ``Recall``         : float
        - ``F1``             : float
        - ``ROC_AUC``        : float
        - ``Inference_Time`` : float — среднее время в секундах на одно изображение
        - ``Parameters``     : int
        - ``Model_Size``     : float — МБ
        - ``Epochs``         : int — фактическое количество выполненных эпох
        - ``Training_Time``  : float — общее время обучения по настенным часам в секундах
    """
    device: torch.device = config.DEVICE
    safe_name: str = model_name.replace("/", "_").replace("-", "_")

    logger.info("=" * 64)
    logger.info("Training model: %s", model_name)
    logger.info("=" * 64)

    # ------------------------------------------------------------------
    # 1. Построение модели
    # ------------------------------------------------------------------
    model: nn.Module = get_model(
        model_name=model_name,
        num_classes=config.NUM_CLASSES,
        pretrained=True,
    )
    model = model.to(device)
    logger.info("Model '%s' moved to device: %s", model_name, device)

    # ------------------------------------------------------------------
    # 2. Оптимизатор и функция потерь
    # ------------------------------------------------------------------
    optimizer: torch.optim.Adam = torch.optim.Adam(
        model.parameters(),
        lr=config.LEARNING_RATE,
    )
    criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss()

    # ------------------------------------------------------------------
    # 3. Колбэки (оба отслеживают F1-меру на валидации — mode="max")
    # ------------------------------------------------------------------
    checkpoint_path: Path = config.SAVED_MODELS_DIR / f"{safe_name}_best.pth"
    early_stopping: EarlyStopping = EarlyStopping(patience=config.PATIENCE, mode="max")
    model_checkpoint: ModelCheckpoint = ModelCheckpoint(save_path=checkpoint_path, mode="max")

    # ------------------------------------------------------------------
    # 4. Цикл обучения
    # ------------------------------------------------------------------
    history: dict[str, list[float]] = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    training_start: float = time.perf_counter()
    epochs_trained: int = 0

    for epoch in range(1, config.NUM_EPOCHS + 1):
        logger.info(
            "Epoch %d/%d — model: %s",
            epoch,
            config.NUM_EPOCHS,
            model_name,
        )

        # Обучаем одну эпоху.
        train_loss, train_acc = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )

        # Оцениваем на валидационной выборке.
        val_loss, val_acc, val_true, val_pred, val_proba = evaluate(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device,
        )

        # Вычисляем F1-меру на валидации для колбэков.
        from sklearn.metrics import f1_score as sklearn_f1_score
        val_f1: float = float(sklearn_f1_score(val_true, val_pred, average="weighted", zero_division=0))

        # Записываем историю.
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        epochs_trained = epoch

        logger.info(
            "  train_loss=%.4f | train_acc=%.4f | val_loss=%.4f | val_acc=%.4f | val_f1=%.4f",
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            val_f1,
        )

        # ModelCheckpoint — сохраняем, если F1-мера на валидации улучшилась.
        model_checkpoint(val_f1, model)

        # EarlyStopping — останавливаем, если F1 не улучшается в течение PATIENCE эпох.
        if early_stopping(val_f1):
            logger.info(
                "Early stopping triggered at epoch %d for model '%s'.",
                epoch,
                model_name,
            )
            break

    training_end: float = time.perf_counter()
    training_time: float = training_end - training_start

    logger.info(
        "Training complete — %d epochs in %.1f s (%.1f min)",
        epochs_trained,
        training_time,
        training_time / 60.0,
    )

    # ------------------------------------------------------------------
    # 5. Загружаем лучший чекпоинт
    # ------------------------------------------------------------------
    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        logger.info("Best checkpoint loaded from '%s'.", checkpoint_path)
    else:
        logger.warning(
            "Checkpoint file '%s' not found — using weights from last epoch.",
            checkpoint_path,
        )

    # ------------------------------------------------------------------
    # 6. Оцениваем на тестовой выборке
    # ------------------------------------------------------------------
    logger.info("Evaluating best model on test set …")
    test_loss, test_acc, y_true, y_pred, y_proba = evaluate(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device,
    )
    logger.info(
        "Test — loss=%.4f | accuracy=%.4f", test_loss, test_acc
    )

    # ------------------------------------------------------------------
    # 7. Вычисляем все метрики классификации
    # ------------------------------------------------------------------
    metrics: dict[str, float] = compute_metrics(
        y_true=y_true,
        y_pred=y_pred,
        y_proba=y_proba,
    )

    # ------------------------------------------------------------------
    # 8. Измеряем время инференса (среднее по 100 одиночным прямым проходам)
    # ------------------------------------------------------------------
    model.eval()
    # Берём одно изображение из тестового загрузчика для использования в качестве зонда.
    probe_images: torch.Tensor
    probe_images, _ = next(iter(test_loader))
    single_image: torch.Tensor = probe_images[0:1].to(device)

    # Прогреваем устройство перед замером времени.
    _WARMUP_RUNS: int = 10
    with torch.no_grad():
        for _ in range(_WARMUP_RUNS):
            _ = model(single_image)

    if device.type == "cuda":
        torch.cuda.synchronize()

    _TIMING_RUNS: int = 100
    inference_start: float = time.perf_counter()
    with torch.no_grad():
        for _ in range(_TIMING_RUNS):
            _ = model(single_image)
            if device.type == "cuda":
                torch.cuda.synchronize()
    inference_end: float = time.perf_counter()

    avg_inference_time: float = (inference_end - inference_start) / _TIMING_RUNS
    logger.info(
        "Inference time (avg over %d runs): %.4f ms",
        _TIMING_RUNS,
        avg_inference_time * 1000.0,
    )

    # ------------------------------------------------------------------
    # 9. Информация о модели (количество параметров и размер)
    # ------------------------------------------------------------------
    model_info: dict[str, Any] = get_model_info(model)
    total_params: int = model_info["total_params"]
    model_size_mb: float = model_info["model_size_mb"]

    # Используем размер чекпоинта на диске, если он доступен, для более точного значения.
    if checkpoint_path.exists():
        disk_size_mb: float = checkpoint_path.stat().st_size / (1024 * 1024)
        logger.info(
            "Checkpoint on-disk size: %.2f MB (parameter estimate: %.2f MB)",
            disk_size_mb,
            model_size_mb,
        )
        model_size_mb = disk_size_mb

    # ------------------------------------------------------------------
    # 10. Строим графики
    # ------------------------------------------------------------------
    plots_dir: Path = output_dir if output_dir is not None else config.PLOTS_DIR

    plot_training_curves(
        history=history,
        model_name=safe_name,
        save_dir=plots_dir,
    )
    logger.info("Training curves saved for '%s'.", model_name)

    plot_confusion_matrix(
        y_true=y_true,
        y_pred=y_pred,
        class_names=list(config.CLASS_NAMES),
        model_name=safe_name,
        save_dir=plots_dir,
    )
    logger.info("Confusion matrix saved for '%s'.", model_name)

    plot_roc_curve(
        y_true=y_true,
        y_proba=y_proba,
        model_name=safe_name,
        save_dir=plots_dir,
    )
    logger.info("ROC curve saved for '%s'.", model_name)

    # ------------------------------------------------------------------
    # 11. Формируем и возвращаем словарь результатов
    # ------------------------------------------------------------------
    inference_time_ms: float = avg_inference_time * 1000.0
    fps: float = 1.0 / avg_inference_time if avg_inference_time > 0 else 0.0

    results: dict[str, Any] = {
        "Architecture": model_name,
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"],
        "ROC AUC": metrics["roc_auc"],
        "Training Time": training_time,
        "Inference Time (ms/image)": inference_time_ms,
        "FPS": fps,
        "Model Size (MB)": model_size_mb,
        "Number of Parameters": total_params,
        "Epochs": epochs_trained,
    }

    logger.info(
        "Results for '%s': acc=%.4f | f1=%.4f | auc=%.4f | "
        "params=%d | size=%.2f MB | epochs=%d | train_time=%.1f s | fps=%.1f",
        model_name,
        results["Accuracy"],
        results["F1"],
        results["ROC AUC"],
        results["Number of Parameters"],
        results["Model Size (MB)"],
        results["Epochs"],
        results["Training Time"],
        results["FPS"],
    )

    return results


# ---------------------------------------------------------------------------
# run_full_pipeline
# ---------------------------------------------------------------------------

def run_full_pipeline(config: type[Config]) -> list[dict[str, Any]]:
    """
    Последовательно обучает все пять архитектур и создаёт все выходные артефакты.

    Шаги
    ----
    1. Фиксирует случайные зерна через ``seed_everything(config.SEED)``.
    2. Скачивает и подготавливает датасет PKLot через
       ``download_and_prepare_dataset(config.DATA_DIR)``.
    3. Создаёт экземпляры ``DataLoader`` с трансформациями для обучения и валидации.
    4. Перебирает все пять имён моделей из ``config.MODEL_NAMES`` и вызывает
       ``train_model()`` для каждой; собирает словари результатов.
    5. Сохраняет сравнительную таблицу по архитектурам (CSV + XLSX) через
       ``save_comparison_table()``.
    6. Генерирует и выводит текстовый анализ через ``analyze_results()``.
    7. Записывает текст анализа в ``config.RESULTS_DIR / "analysis.txt"``.
    8. Возвращает полный список словарей результатов.

    Parameters
    ----------
    config:
        Класс ``Config`` (не экземпляр), содержащий все настройки проекта.

    Returns
    -------
    list[dict[str, Any]]
        Один словарь результатов на каждую обученную модель в порядке,
        определённом ``config.MODEL_NAMES``.
    """
    # ------------------------------------------------------------------
    # 1. Воспроизводимость
    # ------------------------------------------------------------------
    seed_everything(config.SEED)

    # ------------------------------------------------------------------
    # 2. Подготовка датасета
    # ------------------------------------------------------------------
    logger.info("Preparing PKLot dataset …")
    data_root: Path = download_and_prepare_dataset(
        data_dir=config.DATA_DIR,
        seed=config.SEED,
    )

    # ------------------------------------------------------------------
    # 3. DataLoaders
    # ------------------------------------------------------------------
    logger.info("Building DataLoaders …")
    train_loader: DataLoader
    val_loader: DataLoader
    test_loader: DataLoader
    train_loader, val_loader, test_loader = get_data_loaders(
        data_dir=data_root,
        batch_size=config.BATCH_SIZE,
        train_transform=get_train_transforms(),
        val_transform=get_val_transforms(),
    )
    logger.info(
        "DataLoaders ready — train batches: %d | val batches: %d | test batches: %d",
        len(train_loader),
        len(val_loader),
        len(test_loader),
    )

    # ------------------------------------------------------------------
    # 4. Создаём директорию вывода с меткой времени
    # ------------------------------------------------------------------
    timestamp: str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_dir: Path = config.RESULTS_DIR / timestamp
    run_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Results will be saved to '%s'.", run_dir)

    # ------------------------------------------------------------------
    # 5. Последовательно обучаем все модели
    # ------------------------------------------------------------------
    all_results: list[dict[str, Any]] = []

    for model_name in config.MODEL_NAMES:
        logger.info("")
        logger.info("Starting training pipeline for: %s", model_name)

        try:
            result: dict[str, Any] = train_model(
                model_name=model_name,
                train_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                config=config,
                output_dir=run_dir,
            )
            all_results.append(result)
            logger.info("Finished training: %s", model_name)
        except Exception:
            logger.exception(
                "Training failed for model '%s'. Skipping to next model.",
                model_name,
            )

    if not all_results:
        logger.error("No models were successfully trained.")
        return all_results

    # ------------------------------------------------------------------
    # 6. Сохраняем сравнительную таблицу в директорию с меткой времени
    # ------------------------------------------------------------------
    logger.info("Saving comparison table …")
    save_comparison_table(
        results=all_results,
        save_dir=run_dir,
    )
    logger.info(
        "Comparison table saved to '%s'.", run_dir
    )

    # ------------------------------------------------------------------
    # 7. Сохраняем metrics.json
    # ------------------------------------------------------------------
    metrics_path: Path = run_dir / "metrics.json"
    metrics_path.write_text(
        json.dumps(all_results, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )
    logger.info("Metrics JSON saved to '%s'.", metrics_path)

    # ------------------------------------------------------------------
    # 8. Генерируем текстовый анализ
    # ------------------------------------------------------------------
    logger.info("Generating results analysis …")
    analysis_text: str = analyze_results(all_results)
    print(analysis_text)

    # ------------------------------------------------------------------
    # 9. Сохраняем анализ в директорию с меткой времени
    # ------------------------------------------------------------------
    analysis_path: Path = run_dir / "analysis.txt"
    analysis_path.write_text(analysis_text, encoding="utf-8")
    logger.info("Analysis text saved to '%s'.", analysis_path)

    logger.info("Full pipeline complete.  %d models trained.  Output: %s", len(all_results), run_dir)

    return all_results


# ---------------------------------------------------------------------------
# Точка входа
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    pipeline_results: list[dict[str, Any]] = run_full_pipeline(Config)

    logger.info("")
    logger.info("Summary:")
    logger.info("%-20s  %8s  %8s  %8s", "Architecture", "Accuracy", "F1", "ROC AUC")
    logger.info("-" * 52)
    for res in sorted(pipeline_results, key=lambda r: r["F1"], reverse=True):
        logger.info(
            "%-20s  %8.4f  %8.4f  %8.4f",
            res["Architecture"],
            res["Accuracy"],
            res["F1"],
            res["ROC AUC"],
        )


In [ ]:
%%writefile predict.py
"""
predict.py — Инференс на одном изображении для классификатора занятости парковочных мест.

Данный модуль предоставляет три уровня абстракции для запуска инференса:

1. ``load_model()``   — низкий уровень: строит архитектуру и загружает сохранённый
                         state-dict с диска.
2. ``predict_image()`` — средний уровень: принимает путь к изображению, загруженную модель и
                         трансформацию предобработки; возвращает
                         ``PredictionResult``.
3. ``ParkingPredictor`` — высокий уровень: класс-обёртка, владеющий моделью и
                         трансформацией, так что вызывающему коду нужно лишь обращаться к ``.predict()``.

Типичное использование
----------------------
Из Python-кода (например, app.py):

    from predict import ParkingPredictor

    predictor = ParkingPredictor(
        model_path="saved_models/ResNet18_best.pth",
        model_name="ResNet18",
    )
    result = predictor.predict("some_spot.jpg")
    print(result.label)        # "Empty" or "Occupied"
    print(result.confidence)   # e.g. 0.9732
    print(result.probabilities)# {"Empty": 0.0268, "Occupied": 0.9732}

Из командной строки:

    python predict.py --model-path saved_models/ResNet18_best.pth \\
                      --model-name ResNet18 \\
                      --image path/to/spot.jpg
"""

import argparse
import logging
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

from config import Config, get_val_transforms
from models import get_model

# ---------------------------------------------------------------------------
# Логгер уровня модуля
# ---------------------------------------------------------------------------
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Датакласс PredictionResult
# ---------------------------------------------------------------------------

@dataclass
class PredictionResult:
    """
    Контейнер для результата одного инференса на изображении.

    Attributes
    ----------
    label : str
        Предсказанное имя класса, например ``"Empty"`` или ``"Occupied"``.
    confidence : float
        Вероятность softmax для предсказанного класса в диапазоне [0.0, 1.0].
    probabilities : dict[str, float]
        Полное распределение softmax с ключами по именам классов, например
        ``{"Empty": 0.15, "Occupied": 0.85}``.
    inference_time_ms : float
        Время по настенным часам (в миллисекундах), затраченное только на прямой проход
        (без загрузки изображения и предобработки).

    Examples
    --------
    >>> result = PredictionResult(
    ...     label="Occupied",
    ...     confidence=0.9732,
    ...     probabilities={"Empty": 0.0268, "Occupied": 0.9732},
    ...     inference_time_ms=4.7,
    ... )
    >>> result.label
    'Occupied'
    >>> result.confidence
    0.9732
    """

    label: str
    confidence: float
    probabilities: dict[str, float] = field(default_factory=dict)
    inference_time_ms: float = 0.0

    def __str__(self) -> str:
        prob_str = ", ".join(
            f"{name}: {prob:.4f}" for name, prob in self.probabilities.items()
        )
        return (
            f"PredictionResult("
            f"label={self.label!r}, "
            f"confidence={self.confidence:.4f}, "
            f"probabilities={{{prob_str}}}, "
            f"inference_time_ms={self.inference_time_ms:.2f})"
        )


# ---------------------------------------------------------------------------
# load_model
# ---------------------------------------------------------------------------

def load_model(
    model_path: Path,
    model_name: str,
    num_classes: int,
    device: torch.device,
) -> nn.Module:
    """
    Строит архитектуру и восстанавливает веса из сохранённого файла state-dict.

    Функция использует :func:`models.get_model` для построения базовой модели
    (с ``pretrained=False``, чтобы веса ImageNet не скачивались), затем
    загружает дообученные веса из ``model_path`` через
    ``torch.load`` / ``model.load_state_dict``. Модель размещается на
    ``device`` и переводится в режим оценки перед возвратом.

    Parameters
    ----------
    model_path : Path
        Путь к сохранённому файлу ``.pth``, содержащему ``state_dict`` модели.
        Ожидается, что файл был создан через
        ``torch.save(model.state_dict(), path)`` (или аналогичную логику
        ``ModelCheckpoint`` в ``train.py``).
    model_name : str
        Один из пяти канонических идентификаторов архитектур, принимаемых
        :func:`models.get_model`:
        ``"ResNet18"``, ``"DenseNet121"``, ``"EfficientNet-B0"``,
        ``"MobileNetV3-Large"`` или ``"ViT-B/16"``.
    num_classes : int
        Количество выходных классов, для которых обучена модель.
        Используйте ``Config.NUM_CLASSES`` (2), если не переобучали на другом разбиении.
    device : torch.device
        Вычислительное устройство, на которое перемещается модель после загрузки.
        Используйте ``Config.DEVICE`` для автоматического выбора наилучшего устройства.

    Returns
    -------
    nn.Module
        Загруженная модель в режиме оценки на запрошенном устройстве.

    Raises
    ------
    FileNotFoundError
        Если ``model_path`` не существует.
    RuntimeError
        Если ключи state-dict не совпадают с архитектурой модели (например,
        файл ``.pth`` сохранён из другой архитектуры).

    Examples
    --------
    >>> from pathlib import Path
    >>> import torch
    >>> model = load_model(
    ...     model_path=Path("saved_models/ResNet18_best.pth"),
    ...     model_name="ResNet18",
    ...     num_classes=2,
    ...     device=torch.device("cpu"),
    ... )  # doctest: +SKIP
    """
    model_path = Path(model_path)

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model checkpoint not found: {model_path}. "
            "Train the model first (train.py) or provide a valid path."
        )

    logger.info(
        "Loading model '%s' from '%s' onto device '%s' …",
        model_name,
        model_path,
        device,
    )

    # Строим архитектуру без предобученных весов; загрузим собственные.
    model: nn.Module = get_model(model_name, num_classes=num_classes, pretrained=False)

    # Загружаем чекпоинт — маппинг на целевое устройство позволяет загружать
    # GPU-чекпоинты на машине только с CPU.
    state_dict = torch.load(model_path, map_location=device)

    # Некоторые конвейеры обучения оборачивают state-dict в словарь с дополнительными ключами,
    # например {"model_state_dict": ..., "epoch": ..., "val_loss": ...}.
    # Прозрачно обрабатываем оба формата: плоский и вложенный.
    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        logger.debug("Checkpoint contains a nested dict; extracting 'model_state_dict'.")
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    logger.info(
        "Model '%s' loaded successfully (%d parameters, eval mode).",
        model_name,
        sum(p.numel() for p in model.parameters()),
    )
    return model


# ---------------------------------------------------------------------------
# predict_image
# ---------------------------------------------------------------------------

def predict_image(
    image_path: str,
    model: nn.Module,
    transform: transforms.Compose,
    device: torch.device,
    class_names: Sequence[str],
) -> PredictionResult:
    """
    Выполняет один прямой проход и возвращает структурированный результат предсказания.

    Функция обрабатывает загрузку изображения, предобработку, формирование батча, инференс и
    преобразование через softmax. Модель должна уже находиться в режиме оценки; эта
    функция никогда не вызывает ``.train()`` или ``.eval()``.

    Parameters
    ----------
    image_path : str
        Путь в файловой системе к файлу изображения. Принимается любой формат,
        поддерживаемый Pillow (JPEG, PNG, BMP, TIFF, …).
    model : nn.Module
        Загруженная модель в режиме оценки, как возвращает :func:`load_model`.
    transform : transforms.Compose
        Трансформация предобработки, применяемая перед прямым проходом. В
        продакшене передавайте результат ``get_val_transforms()`` из
        ``config.py``.
    device : torch.device
        Устройство для перемещения тензора изображения (должно совпадать с устройством модели).
    class_names : Sequence[str]
        Упорядоченный список меток классов, соответствующих выходным логитам модели.
        Для данного проекта передавайте ``Config.CLASS_NAMES``
        (``["Empty", "Occupied"]``).

    Returns
    -------
    PredictionResult
        Экземпляр датакласса, содержащий предсказанную метку, уверенность,
        полное распределение вероятностей и время прямого прохода.

    Raises
    ------
    FileNotFoundError
        Если ``image_path`` не указывает на существующий файл.
    OSError
        Если Pillow не может открыть файл (повреждённый или неподдерживаемый формат).

    Examples
    --------
    >>> from config import Config, get_val_transforms
    >>> from predict import load_model, predict_image
    >>> import torch
    >>> model = load_model(
    ...     Path("saved_models/ResNet18_best.pth"),
    ...     "ResNet18", 2, torch.device("cpu")
    ... )  # doctest: +SKIP
    >>> result = predict_image(
    ...     "test_spot.jpg", model, get_val_transforms(),
    ...     torch.device("cpu"), Config.CLASS_NAMES,
    ... )  # doctest: +SKIP
    """
    image_path_obj = Path(image_path)
    if not image_path_obj.exists():
        raise FileNotFoundError(f"Image file not found: {image_path}")

    logger.debug("Opening image '%s'.", image_path)

    # Загружаем и конвертируем в RGB, чтобы grayscale и RGBA изображения обрабатывались
    # единообразно для всех пяти архитектур.
    pil_image: Image.Image = Image.open(image_path_obj).convert("RGB")

    # Применяем предобработку на время валидации (resize → centre-crop → normalise).
    tensor: torch.Tensor = transform(pil_image)  # shape: (C, H, W)

    # Добавляем размерность батча, необходимую для nn.Module: (1, C, H, W).
    tensor = tensor.unsqueeze(0).to(device)

    # Прямой проход — отключаем вычисление градиентов для ускорения инференса.
    t_start = time.perf_counter()
    with torch.no_grad():
        logits: torch.Tensor = model(tensor)  # shape: (1, num_classes)
    t_end = time.perf_counter()
    inference_ms: float = (t_end - t_start) * 1_000.0

    # Преобразуем логиты в вероятности через softmax по размерности классов.
    probs: torch.Tensor = F.softmax(logits, dim=1).squeeze(0)  # shape: (num_classes,)

    # Определяем победивший класс.
    predicted_idx: int = int(probs.argmax().item())
    confidence: float = float(probs[predicted_idx].item())
    label: str = class_names[predicted_idx]

    # Формируем полный маппинг вероятностей.
    probabilities: dict[str, float] = {
        name: float(probs[i].item()) for i, name in enumerate(class_names)
    }

    result = PredictionResult(
        label=label,
        confidence=confidence,
        probabilities=probabilities,
        inference_time_ms=inference_ms,
    )

    logger.info(
        "Prediction for '%s': label=%s, confidence=%.4f, time=%.2f ms.",
        image_path,
        label,
        confidence,
        inference_ms,
    )
    return result


# ---------------------------------------------------------------------------
# ParkingPredictor — высокоуровневый класс-обёртка
# ---------------------------------------------------------------------------

class ParkingPredictor:
    """
    Высокоуровневая, stateful обёртка для инференса классификации парковочных мест.

    ``ParkingPredictor`` загружает модель один раз в ``__init__`` и предоставляет
    простой метод ``.predict()``, так что вызывающему коду не нужно управлять моделью,
    устройством или трансформацией самостоятельно. Этот класс используется в ``app.py``
    (интерфейс Streamlit).

    Parameters
    ----------
    model_path : str | Path
        Путь к сохранённому чекпоинту ``.pth``.
    model_name : str
        Идентификатор архитектуры, один из ``Config.MODEL_NAMES``.
    num_classes : int, optional
        Количество выходных классов. По умолчанию ``Config.NUM_CLASSES`` (2).
    device : torch.device | None, optional
        Целевое вычислительное устройство. Если ``None`` (по умолчанию), используется
        ``Config.DEVICE`` (CUDA > MPS > CPU).

    Attributes
    ----------
    model_name : str
        Имя загруженной архитектуры.
    model_path : Path
        Разрешённый путь к файлу чекпоинта.
    device : torch.device
        Устройство, на которое загружена модель.
    class_names : tuple[str, ...]
        Метки классов из ``Config.CLASS_NAMES``.

    Examples
    --------
    >>> predictor = ParkingPredictor(
    ...     model_path="saved_models/ResNet18_best.pth",
    ...     model_name="ResNet18",
    ... )  # doctest: +SKIP
    >>> result = predictor.predict("parking_spot.jpg")  # doctest: +SKIP
    >>> print(result.label, result.confidence)          # doctest: +SKIP
    Occupied 0.9912
    """

    def __init__(
        self,
        model_path: str | Path,
        model_name: str,
        num_classes: int = Config.NUM_CLASSES,
        device: torch.device | None = None,
    ) -> None:
        """
        Загружает модель с диска и подготавливает трансформацию предобработки.

        Parameters
        ----------
        model_path : str | Path
            Путь к сохранённому чекпоинту (файл ``.pth``).
        model_name : str
            Имя архитектуры для построения правильного скелета модели.
        num_classes : int, optional
            Количество классов. По умолчанию ``Config.NUM_CLASSES`` (2).
        device : torch.device | None, optional
            Устройство для инференса. По умолчанию ``Config.DEVICE``.
        """
        self.model_path: Path = Path(model_path)
        self.model_name: str = model_name
        self.device: torch.device = device if device is not None else Config.DEVICE
        self.class_names: tuple[str, ...] = Config.CLASS_NAMES
        self._num_classes: int = num_classes

        logger.info(
            "Initialising ParkingPredictor — arch=%s, device=%s.",
            model_name,
            self.device,
        )

        # Загружаем модель один раз; она будет использоваться повторно при каждом последующем вызове.
        self._model: nn.Module = load_model(
            model_path=self.model_path,
            model_name=self.model_name,
            num_classes=self._num_classes,
            device=self.device,
        )

        # Трансформация на время валидации (без аугментации, детерминированная).
        self._transform: transforms.Compose = get_val_transforms()

        logger.info("ParkingPredictor ready.")

    def predict(self, image_path: str | Path) -> PredictionResult:
        """
        Классифицирует одно изображение парковочного места как ``"Empty"`` или ``"Occupied"``.

        Parameters
        ----------
        image_path : str | Path
            Путь в файловой системе к классифицируемому изображению.

        Returns
        -------
        PredictionResult
            Датакласс с полями ``label``, ``confidence``, ``probabilities`` и
            ``inference_time_ms``.

        Raises
        ------
        FileNotFoundError
            Если ``image_path`` не существует.
        OSError
            Если файл изображения не может быть открыт Pillow.

        Examples
        --------
        >>> result = predictor.predict("lot_A_007.jpg")  # doctest: +SKIP
        >>> result.label
        'Empty'
        >>> result.confidence > 0.5
        True
        """
        return predict_image(
            image_path=str(image_path),
            model=self._model,
            transform=self._transform,
            device=self.device,
            class_names=self.class_names,
        )

    def __repr__(self) -> str:
        return (
            f"ParkingPredictor("
            f"model_name={self.model_name!r}, "
            f"model_path={self.model_path!r}, "
            f"device={self.device!r})"
        )


# ---------------------------------------------------------------------------
# Точка входа командной строки
# ---------------------------------------------------------------------------

def _build_arg_parser() -> argparse.ArgumentParser:
    """Возвращает парсер аргументов для точки входа CLI."""
    parser = argparse.ArgumentParser(
        prog="predict",
        description=(
            "Classify a single parking-spot image as Empty or Occupied "
            "using one of the five trained models."
        ),
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--model-path",
        required=True,
        metavar="PATH",
        help="Path to the saved .pth checkpoint file.",
    )
    parser.add_argument(
        "--model-name",
        required=True,
        choices=list(Config.MODEL_NAMES),
        metavar="NAME",
        help=(
            "Architecture name.  One of: "
            + ", ".join(f'"{n}"' for n in Config.MODEL_NAMES)
            + "."
        ),
    )
    parser.add_argument(
        "--image",
        required=True,
        metavar="PATH",
        help="Path to the parking-spot image to classify.",
    )
    parser.add_argument(
        "--device",
        default=str(Config.DEVICE),
        metavar="DEVICE",
        help="Torch device string, e.g. 'cpu', 'cuda', 'mps'.",
    )
    return parser


def main(argv: list[str] | None = None) -> None:
    """
    Точка входа CLI для инференса на одном изображении.

    Парсит аргументы командной строки, загружает указанную модель, классифицирует
    переданное изображение и выводит результат в stdout.

    Parameters
    ----------
    argv : list[str] | None, optional
        Список аргументов для тестирования. По умолчанию ``sys.argv[1:]`` при
        значении ``None``.

    Examples
    --------
    .. code-block:: bash

        python predict.py \\
            --model-path saved_models/ResNet18_best.pth \\
            --model-name ResNet18 \\
            --image test_spot.jpg
    """
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    parser = _build_arg_parser()
    args = parser.parse_args(argv)

    device = torch.device(args.device)

    predictor = ParkingPredictor(
        model_path=args.model_path,
        model_name=args.model_name,
        device=device,
    )

    result = predictor.predict(args.image)

    print("-" * 48)
    print(f"  Image      : {args.image}")
    print(f"  Model      : {args.model_name}")
    print(f"  Label      : {result.label}")
    print(f"  Confidence : {result.confidence:.4f} ({result.confidence * 100:.2f} %)")
    print("  Probabilities:")
    for class_name, prob in result.probabilities.items():
        print(f"    {class_name:<12} {prob:.4f}")
    print(f"  Latency    : {result.inference_time_ms:.2f} ms")
    print("-" * 48)


if __name__ == "__main__":
    main()


## Обучение моделей

Запускаем полный пайплайн обучения всех пяти архитектур.

In [ ]:
# Настройка логирования
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)

# Импорт и запуск обучения
from config import Config, seed_everything
from train import run_full_pipeline

seed_everything(Config.SEED)
results = run_full_pipeline(Config)

## Результаты

In [ ]:
# Таблица сравнения моделей
import pandas as pd
import os

# Находим последнюю папку с результатами
results_dirs = sorted([d for d in os.listdir('results') if os.path.isdir(os.path.join('results', d))])
if results_dirs:
    latest_dir = os.path.join('results', results_dirs[-1])
    df = pd.read_csv(os.path.join(latest_dir, 'comparison.csv'))
    print(f"Результаты из: {latest_dir}")
    print("Сравнение архитектур:")
    display(df)

In [ ]:
# Отображение графиков
import glob
from IPython.display import Image, display
import os

results_dirs = sorted([d for d in os.listdir('results') if os.path.isdir(os.path.join('results', d))])
if results_dirs:
    latest_dir = os.path.join('results', results_dirs[-1])
    for path in sorted(glob.glob(os.path.join(latest_dir, '*.png'))):
        print(f"\n{path}")
        display(Image(filename=path, width=700))

In [ ]:
# Анализ результатов
import os

results_dirs = sorted([d for d in os.listdir('results') if os.path.isdir(os.path.join('results', d))])
if results_dirs:
    latest_dir = os.path.join('results', results_dirs[-1])
    with open(os.path.join(latest_dir, 'analysis.txt'), 'r', encoding='utf-8') as f:
        print(f.read())

## Тестирование модели

Проверяем лучшую модель на тестовом изображении.

In [ ]:
# Тестирование лучшей модели на примере
import os
import pandas as pd
from IPython.display import Image, display
from config import Config
from predict import ParkingPredictor

# Определяем лучшую модель из таблицы
results_dirs = sorted([d for d in os.listdir('results') if os.path.isdir(os.path.join('results', d))])
if results_dirs:
    latest_dir = os.path.join('results', results_dirs[-1])
    df = pd.read_csv(os.path.join(latest_dir, 'comparison.csv'))
    best_arch = df.sort_values('F1', ascending=False).iloc[0]['Architecture']
    print(f"Лучшая модель: {best_arch}")

    # Загружаем модель
    safe_name = best_arch.replace("/", "_").replace("-", "_")
    model_path = f"saved_models/{safe_name}_best.pth"
    predictor = ParkingPredictor(model_path=model_path, model_name=best_arch)

    # Берём случайное изображение из тестовой выборки
    test_dir = os.path.join(str(Config.DATA_DIR), 'test')
    for class_name in os.listdir(test_dir):
        class_dir = os.path.join(test_dir, class_name)
        if os.path.isdir(class_dir):
            images = [f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.png'))]
            if images:
                img_path = os.path.join(class_dir, images[0])
                result = predictor.predict(img_path)
                print(f"\nФайл: {images[0]}")
                print(f"Реальный класс: {class_name}")
                print(f"Предсказание: {result.label}")
                print(f"Уверенность: {result.confidence:.2%}")
                display(Image(filename=img_path, width=300))
                break